# Malicious Comment Legal Risk Insurance Service Project
## Complete Analysis Pipeline

### Workflow

```text
[1] Load and merge crawled data
        ↓
[2] Explore the dataset structure
        ↓
[3] Analyze temporal patterns and engagement
        ↓
[4] Analyze comment text
        ↓
[5] Review first-stage LLM screening results
        ↓
[6] Train and evaluate models on manually labeled data
        ↓
[7] Combine legal-element scores into offense probabilities
        ↓
[8] Aggregate comments into video-level incidents
        ↓
[9] Calculate premiums and visualize results
```

> Korean source labels, legal-category keys, regular expressions, and workbook sheet names are retained where required for compatibility with the original data files.


---
## 0. Environment Setup


In [ ]:
import csv
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib import font_manager
from collections import Counter
import re

warnings.filterwarnings('ignore')

# Configure a Korean-capable font on macOS.
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Configure project paths.
BASE = Path('YOUR_PROJECT_DIRECTORY')
RISK_DIR = BASE / 'Risk_final'

print('Environment setup complete')

---
## 1. Load the Source Data

### File structure

| File | Subjects | Notes |
|---|---|---|
| `comments_all.csv` | F + K + S | Consolidated file containing all comment IDs from the individual files |
| `f_comments_all.csv` | F | Already included in `comments_all.csv`; no separate load required |
| `k_comments_all.csv` | K | Already included in `comments_all.csv` |
| `s_all.csv` | S | Already included in `comments_all.csv` |

Only `comments_all.csv` is loaded because it is already consolidated. Merging the individual files again would create extensive overlap in `comment_id` and cause excessive data loss during deduplication. Verification showed that all 78,518 F, 53,566 K, and 54,461 S records were included.

### CSV parsing note

Some comments contain commas or line breaks that can break a standard `pandas.read_csv` call. The loader therefore uses Python's `csv.reader` and reconstructs rows whose comment field contains additional delimiters.


In [ ]:
def load_csv_safe(path, expected_cols=None):
    rows = []
    with open(path, encoding='utf-8-sig', newline='') as f:
        reader = csv.reader(f)
        header = next(reader)
        n = expected_cols or len(header)
        for row in reader:
            if len(row) == n:
                rows.append(row)
            elif len(row) > n:
                # Domain-specific processing step; Korean schema keys are retained below.
                extra = len(row) - n
                fixed = row[:3] + [','.join(row[3:3 + extra + 1])] + row[3 + extra + 1:]
                rows.append(fixed)
    return pd.DataFrame(rows, columns=header[:n])


df = load_csv_safe(BASE / 'comments_all.csv')
print(f'Load complete: {len(df):,} rows')
print(f'Columns: {df.columns.tolist()}')

In [ ]:
SUBJECT_KEYWORDS = {
    'Subject_A': ['REDACTED_SUBJECT_A'],
    'Subject_B': ['REDACTED_SUBJECT_B'],
    'Subject_C': ['REDACTED_SUBJECT_C'],
}

def tag_person(title):
    t = str(title).lower()
    for person, kws in SUBJECT_KEYWORDS.items():
        if any(kw.lower() in t for kw in kws):
            return person
    return 'Other'

# Normalize column types.
df['like_count'] = pd.to_numeric(df['like_count'], errors='coerce').fillna(0).astype(int)
df['published_at'] = pd.to_datetime(df['published_at'], errors='coerce', utc=True)
if 'is_malicious' not in df.columns and 'is_profanity' in df.columns:
    df = df.rename(columns={'is_profanity': 'is_malicious'})
elif 'is_profanity' in df.columns:
    df['is_malicious'] = df['is_malicious'].fillna(df['is_profanity'])
    df = df.drop(columns=['is_profanity'])
df['is_malicious'] = df['is_malicious'].astype(str).str.upper().map({'TRUE': True, 'FALSE': False})

# Assign subject labels.
df['person'] = df['video_title'].map(tag_person)

# Remove duplicate comment IDs introduced across crawl batches.
# Keep the first occurrence to avoid distorting EDA counts.
before = len(df)
df = df.drop_duplicates(subset='comment_id', keep='first').reset_index(drop=True)
print(f'Total rows        : {before:,}')
print(f'After deduplication    : {len(df):,}  (removed: {before - len(df):,} records)')
print()
print('comments by subject:')
print(df['person'].value_counts())

df.head(3)

F videos often omit her name from their titles and may feature other influencers or celebrities, making exact subject classification difficult.

---
## 2. EDA — Dataset Structure


In [ ]:
print(f'Total comments  : {len(df):,}')
print(f'Unique videos: {df["video_id"].nunique():,}')
print(f'Unique authors : {df["author"].nunique():,}')
print(df.isnull().sum())

In [ ]:
# Count comments by subject.
person_cnt = df['person'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(person_cnt.index, person_cnt.values, color=['#4C72B0','#DD8452','#55A467','#C44E52'])
axes[0].set_title('comments by subject')
axes[0].set_ylabel('comments')
for i, v in enumerate(person_cnt.values):
    axes[0].text(i, v + 500, f'{v:,}', ha='center', fontsize=9)

# Count unique videos by subject.
video_cnt = df.groupby('person')['video_id'].nunique().sort_values(ascending=False)
axes[1].bar(video_cnt.index, video_cnt.values, color=['#4C72B0','#DD8452','#55A467','#C44E52'])
axes[1].set_title('Unique videos by subject')
axes[1].set_ylabel('Video Count')
for i, v in enumerate(video_cnt.values):
    axes[1].text(i, v + 1, f'{v:,}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print('\nAverage comments per video by subject:')
print((person_cnt / video_cnt).round(1).rename('average_comments_per_video'))

In [ ]:
# Analyze the like-count distribution.
like_raw = df.loc[:, 'like_count']

if isinstance(like_raw, pd.DataFrame):
    like_raw = like_raw.iloc[:, 0]

lc = (
    like_raw.astype(str)
    .str.replace(',', '', regex=False)
    .pipe(pd.to_numeric, errors='coerce')
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)

print('[Like-count summary statistics]')
print(lc.describe().apply(lambda x: f'{x:,.1f}'))
print(f'\nShare of comments with zero likes: {(lc == 0).mean():.1%}')
print(f'comments with at least 10 likes: {(lc >= 10).sum():,}')

fig, ax = plt.subplots(figsize=(12, 5))

upper = lc.quantile(0.99)
clipped = np.clip(lc.to_numpy(dtype=float), 0, upper)

bin_edges = np.linspace(0, upper, 51)
counts = np.zeros(50, dtype=int)

idx = np.searchsorted(bin_edges, clipped, side='right') - 1
idx = np.clip(idx, 0, 49)

for i in idx:
    counts[i] += 1

bars = ax.bar(
    bin_edges[:-1],
    counts,
    width=np.diff(bin_edges),
    align='edge',
    color='#4C72B0',
    alpha=0.8
)

# Add counts above bars.
for bar, count in zip(bars, counts):
    if count > 0:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f'{count:,}',
            ha='center',
            va='bottom',
            fontsize=8,
            rotation=90
        )

ax.set_title('Like-count distribution (top 1% clipped)')
ax.set_xlabel('Likes')
ax.set_ylabel('comments')
ax.margins(y=0.15)
plt.tight_layout()
plt.show()



---
## 3. EDA — Temporal and Engagement Analysis


In [ ]:
# Calculate monthly comment trends.
df_time = df.dropna(subset=['published_at']).copy()
df_time['ym'] = df_time['published_at'].dt.to_period('M')

monthly = df_time.groupby(['ym', 'person']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(14, 5))
monthly.plot(ax=ax, linewidth=2)
ax.set_title('Monthly comment trend by subject')
ax.set_xlabel('Year-month')
ax.set_ylabel('comments')
ax.legend(title='Subject')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Analyze weekday and hourly distributions.
df_time['hour'] = df_time['published_at'].dt.hour
df_time['weekday'] = df_time['published_at'].dt.day_name()

WEEKDAY_ORDER = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
WEEKDAY_LABELS    = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

hour_cnt = df_time['hour'].value_counts().sort_index()
axes[0].bar(hour_cnt.index, hour_cnt.values, color='#4C72B0')
axes[0].set_title('Comment frequency by hour')
axes[0].set_xlabel('Hour (UTC)')
axes[0].set_ylabel('comments')
axes[0].set_xticks(range(0, 24, 3))

wd_cnt = df_time['weekday'].value_counts().reindex(WEEKDAY_ORDER, fill_value=0)
axes[1].bar(WEEKDAY_LABELS, wd_cnt.values, color='#DD8452')
axes[1].set_title('Comment frequency by weekday')
axes[1].set_xlabel('Weekday')
axes[1].set_ylabel('comments')

plt.tight_layout()
plt.show()

In [ ]:
# Identify the top 15 videos by comment count.
top_videos = (
    df.groupby(['video_id', 'video_title', 'person'])
    .size()
    .reset_index(name='comment_count')
    .sort_values('comment_count', ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(12, 6))
labels = [f"[{r['person']}] {str(r['video_title'])[:30]}..." for _, r in top_videos.iterrows()]
ax.barh(labels[::-1], top_videos['comment_count'].values[::-1], color='#55A467')
ax.set_title('Top 15 videos by comment count')
ax.set_xlabel('comments')
plt.tight_layout()
plt.show()

---
## 4. EDA — Text Analysis


In [ ]:
# Analyze comment lengths.
df['text_len'] = df['text'].astype(str).str.len()

print('[Comment-length summary statistics]')
print(df['text_len'].describe().apply(lambda x: f'{x:.1f}'))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Calculate histogram bins directly.
text_len_clipped = df['text_len'].clip(upper=300).to_numpy(dtype=float)

bin_edges = np.linspace(0, 300, 61)
counts = np.zeros(60, dtype=int)

idx = np.searchsorted(bin_edges, text_len_clipped, side='right') - 1
idx = np.clip(idx, 0, 59)

for i in idx:
    counts[i] += 1

bars = axes[0].bar(
    bin_edges[:-1],
    counts,
    width=np.diff(bin_edges),
    align='edge',
    color='#4C72B0',
    alpha=0.8
)

# Domain-specific processing step; Korean schema keys are retained below.
for bar, count in zip(bars, counts):
    if count > 0:
        axes[0].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f'{count:,}',
            ha='center',
            va='bottom',
            fontsize=7,
            rotation=90
        )

axes[0].set_title('Comment-length distribution (top 300 characters)')
axes[0].set_xlabel('Characters')
axes[0].set_ylabel('Frequency')
axes[0].margins(y=0.15)

# Plot comment lengths by subject.
persons = df['person'].dropna().unique()
data_box = [
    df.loc[df['person'] == p, 'text_len'].clip(upper=300).to_numpy(dtype=float)
    for p in persons
]

axes[1].boxplot(data_box, tick_labels=persons, patch_artist=True)
axes[1].set_title('Comment-length distribution by subject')
axes[1].set_ylabel('Characters')

plt.tight_layout()
plt.show()


In [ ]:
# Analyze frequent words using Korean tokens.
def extract_words(series, min_len=2):
    all_words = []
    for text in series.dropna().astype(str):
        words = re.findall(r'[가-힣]{%d,}' % min_len, text)
        all_words.extend(words)
    return Counter(all_words)

# Remove common words that add little analytical value.
STOPWORDS = {'그냥', '진짜', '근데', '이게', '이거', '이건', '저도', '저는', '제가',
             '그게', '그거', '그건', '이런', '저런', '그런', '이렇게', '저렇게', '그렇게',
             '하지만', '그리고', '그래서', '근데', '근거', '이제', '여기', '저기', '거기',
             '아직', '정말', '너무', '되게', '완전', '계속', '이미', '다시', '그냥', '좀더'}

word_cnt = extract_words(df['text'])
filtered = {w: c for w, c in word_cnt.items() if w not in STOPWORDS and len(w) >= 2}
top_words = sorted(filtered.items(), key=lambda x: -x[1])[:30]

fig, ax = plt.subplots(figsize=(12, 6))
words, cnts = zip(*top_words)
ax.barh(list(words)[::-1], list(cnts)[::-1], color='#4C72B0')
ax.set_title('Top 30 words across all comments (stopwords removed)')
ax.set_xlabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
# Detect profanity signals, repeated expressions, emoji, and related patterns.
PATTERNS = {
    'direct_profanity': r'[시씨][발팔]|[개걔][새색][끼기]|ㅅㅂ|ㅂㅅ|ㄱㅅ|ㅁㅊ',
    'obfuscated_profanity': r'병\s*[1i일]\s*신|씨\s*[발팔]|[ㅅ씩][ㅂ발]',
    'jamo_emphasis': r'[ㄱ-ㅎ]{2,}',
    'repetition_emphasis': r'(.)(\1{2,})',  # Domain-specific Korean-language processing rule.
    'threat_expression': r'죽|꺼져|사라져|신고|고소|찌질|패',
    'sexual_expression': r'몸매|섹|야하|치마|속옷|비키니',
}

for name, pat in PATTERNS.items():
    df[f'flag_{name}'] = df['text'].astype(str).str.contains(pat, case=False, regex=True)

flag_cols = [c for c in df.columns if c.startswith('flag_')]
flag_summary = df[flag_cols].sum().rename(lambda x: x.replace('flag_', ''))

print('=== comments matching each pattern ===')
for k, v in flag_summary.items():
    print(f'  {k:15s}: {v:,} records ({v/len(df):.1%})')

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(flag_summary.index, flag_summary.values, color='#C44E52')
ax.set_title('comments matching each pattern')
ax.set_ylabel('comments')
plt.tight_layout()
plt.show()

---
## 5. First-Stage Filtering Analysis (LLM Screening)

This section analyzes comments selected by the LLM as candidates for legal review.

- **lawsuit_review_level**: LOW / MEDIUM / HIGH
- **matched_categories**: detected legal-risk categories
- **matched_terms**: detected expressions


In [ ]:
# Load first-stage filtering results.
df_filter = load_csv_safe(BASE / 'comments_lawsuit_review_candidates.csv')

# Normalize data types.
df_filter['like_count'] = pd.to_numeric(df_filter.get('like_count', 0), errors='coerce').fillna(0).astype(int)
df_filter['person'] = df_filter['video_title'].map(tag_person)

print(f'comments passing the first-stage filter: {len(df_filter):,} records')
print(f'Share of all comments: {len(df_filter)/len(df):.1%}')
print()
print('lawsuit_review_level distribution:')
print(df_filter['lawsuit_review_level'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot the risk-level distribution.
level_cnt = df_filter['lawsuit_review_level'].value_counts()
colors_level = {'HIGH': '#C44E52', 'MEDIUM': '#DD8452', 'LOW': '#55A467'}
bar_colors = [colors_level.get(l, '#4C72B0') for l in level_cnt.index]
axes[0].bar(level_cnt.index, level_cnt.values, color=bar_colors)
axes[0].set_title('comments by risk level')
axes[0].set_ylabel('comments')
for i, v in enumerate(level_cnt.values):
    axes[0].text(i, v + 10, f'{v:,}', ha='center', fontsize=9)

# Cross-tabulate subjects and risk levels.
cross = df_filter.groupby(['person', 'lawsuit_review_level']).size().unstack(fill_value=0)
cross_pct = cross.div(cross.sum(axis=1), axis=0) * 100
cross_pct[['HIGH', 'MEDIUM', 'LOW']].plot(
    kind='bar', stacked=True, ax=axes[1],
    color=['#C44E52', '#DD8452', '#55A467']
)
axes[1].set_title('Risk-level composition by subject (%)')
axes[1].set_ylabel('Share (%)')
axes[1].legend(title='Level')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Analyze matched categories.
# Split comma-separated category values.
all_cats = []
for cats in df_filter['matched_categories'].dropna().astype(str):
    all_cats.extend([c.strip() for c in cats.split(',') if c.strip()])

cat_cnt = Counter(all_cats)
top_cats = cat_cnt.most_common(15)

labels_cat, vals_cat = zip(*top_cats)
fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(list(labels_cat)[::-1], list(vals_cat)[::-1], color='#4C72B0')
ax.set_title('Top 15 detected categories')
ax.set_xlabel('Frequency')
plt.tight_layout()
plt.show()

---
## 6.1 Manually Labeled Data Analysis

This section analyzes comments that passed the first-stage filter and were manually reviewed. The labeled observations are used as training data for the seven-indicator scoring models.


In [ ]:
df_labeled = pd.read_excel('Risk_final/comments_lawsuit_indexing_all_filled.xlsx')
df_labeled['person'] = df_labeled['video_title'].map(tag_person)

print(f'Labeled data: {len(df_labeled):,} records')
print(f'Columns: {df_labeled.columns.tolist()}')
print()
df_labeled.head(3)

In [ ]:
colors_level = {
    'Low': '#55A467',
    'Medium': '#DD8452',
    'High': '#C44E52'
}

In [ ]:
# Domain-specific processing step; Korean schema keys are retained below.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

level_cnt2 = df_labeled['lawsuit_review_level'].value_counts()
bar_colors2 = [colors_level.get(l, '#4C72B0') for l in level_cnt2.index]
axes[0].bar(level_cnt2.index, level_cnt2.values, color=bar_colors2)
axes[0].set_title('Labeled sample — risk-level distribution')
axes[0].set_ylabel('comments')
for i, v in enumerate(level_cnt2.values):
    axes[0].text(i, v + 1, str(v), ha='center', fontsize=10)

# Domain-specific processing step; Korean schema keys are retained below.
person_cnt2 = df_labeled['person'].value_counts()
axes[1].pie(person_cnt2.values, labels=person_cnt2.index, autopct='%1.1f%%',
            colors=['#4C72B0','#DD8452','#55A467','#C44E52'])
axes[1].set_title('Labeled sample composition by subject')

plt.tight_layout()
plt.show()

In [ ]:
# Domain-specific processing step; Korean schema keys are retained below.
df_labeled['text_len'] = df_labeled['text'].astype(str).str.len()

print('=== Labeled-comment length ===')
print(df_labeled.groupby('lawsuit_review_level')['text_len'].describe().round(1))

# Domain-specific processing step; Korean schema keys are retained below.
fig, ax = plt.subplots(figsize=(10, 4))
for lvl, color in [('HIGH', '#C44E52'), ('MEDIUM', '#DD8452'), ('LOW', '#55A467')]:
    vals = df_labeled[df_labeled['lawsuit_review_level'] == lvl]['text_len'].clip(upper=500)
    if len(vals) > 0:
        ax.hist(vals, bins=30, alpha=0.6, label=lvl, color=color)
ax.set_title('Comment length by risk level')
ax.set_xlabel('Characters')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.show()

## 6.2 Model Comparison — Five Algorithms × Seven Indicators

Five algorithms predict seven legal-risk indicators—profanity intensity, factual assertion, sexual expression, threat, repetition, specificity, and publicity—on a 1–3 scale. Performance is compared using five-fold cross-validation.

Algorithms:

- **Multinomial NB**: word-frequency probabilistic baseline
- **Complement NB**: Naive Bayes variant designed for imbalanced data
- **Logistic OvR**: one-vs-rest logistic classifier
- **Ridge OvR**: L2-regularized one-vs-rest linear classifier
- **Nearest Centroid**: classifier based on class-mean vectors

The labeled sample is relatively small, so fine-tuning a large pretrained language model would carry substantial overfitting risk. These five classifiers are implemented with NumPy to reduce environment dependencies and improve reproducibility.


## 6.3 Tokenizer


In [ ]:
# Domain-specific processing step; Korean schema keys are retained below.
# Domain-specific processing step; Korean schema keys are retained below.
# Domain-specific processing step; Korean schema keys are retained below.
# Domain-specific processing step; Korean schema keys are retained below.
# Domain-specific processing step; Korean schema keys are retained below.

import re
import numpy as np
from collections import Counter
from typing import Sequence

WORD_RE = re.compile(r'[가-힣ㄱ-ㅎㅏ-ㅣa-zA-Z0-9]+')
LAUGH_RE = re.compile(r'[ㅋㅎ]{2,}')
CRY_RE   = re.compile(r'[ㅠㅜ]{2,}')
EXCL_RE  = re.compile(r'[!?~]{2,}')


def make_tokens(text: str) -> list:
    if not isinstance(text, str):
        return []
    text = text.lower()
    tokens = []
    # Domain-specific processing step; Korean schema keys are retained below.
    words = WORD_RE.findall(text)
    tokens.extend(words)
    # Domain-specific processing step; Korean schema keys are retained below.
    for i in range(len(words) - 1):
        tokens.append(f'__bg__{words[i]}_{words[i+1]}')
    # Domain-specific processing step; Korean schema keys are retained below.
    han_chunks = re.findall(r'[가-힣ㄱ-ㅎㅏ-ㅣ]+', text)
    for chunk in han_chunks:
        for n in range(2, 6):
            for i in range(len(chunk) - n + 1):
                tokens.append(f'__c{n}__{chunk[i:i+n]}')
    # Domain-specific processing step; Korean schema keys are retained below.
    if LAUGH_RE.search(text):
        tokens.append('__laugh__')
    if CRY_RE.search(text):
        tokens.append('__cry__')
    if EXCL_RE.search(text):
        tokens.append('__excl__')
    return tokens


class TokenVectorizer:
    """Lightweight vectorizer that learns a vocabulary from training data and produces TF-IDF features."""

    def __init__(self, max_features: int = 10000, min_df: int = 1):
        self.max_features = max_features
        self.min_df = min_df
        self.vocab_ = None
        self.idf_ = None

    def fit(self, texts: Sequence[str]) -> 'TokenVectorizer':
        df_counter = Counter()
        for t in texts:
            df_counter.update(set(make_tokens(t)))
        # Domain-specific processing step; Korean schema keys are retained below.
        items = [(tok, df) for tok, df in df_counter.items() if df >= self.min_df]
        items.sort(key=lambda x: -x[1])
        items = items[: self.max_features]
        self.vocab_ = {tok: i for i, (tok, _) in enumerate(items)}
        n_docs = len(texts)
        idf = np.zeros(len(self.vocab_), dtype=np.float32)
        for tok, i in self.vocab_.items():
            idf[i] = np.log((1 + n_docs) / (1 + df_counter[tok])) + 1.0
        self.idf_ = idf
        return self

    def transform_counts(self, texts: Sequence[str]) -> np.ndarray:
        if self.vocab_ is None:
            raise RuntimeError('Call fit() before transform()')
        n = len(texts)
        m = len(self.vocab_)
        x = np.zeros((n, m), dtype=np.float32)
        for i, t in enumerate(texts):
            for tok in make_tokens(t):
                j = self.vocab_.get(tok)
                if j is not None:
                    x[i, j] += 1.0
        return x

    def transform_tfidf(self, texts: Sequence[str]) -> np.ndarray:
        counts = self.transform_counts(texts)
        # Domain-specific processing step; Korean schema keys are retained below.
        tfidf = counts * self.idf_[None, :]
        norms = np.linalg.norm(tfidf, axis=1, keepdims=True)
        norms[norms == 0] = 1.0
        return tfidf / norms


print('Tokenizer definition complete')

## 6.4 Define the Five Classifiers


In [ ]:
# Domain-specific processing step; Korean schema keys are retained below.
# Domain-specific processing step; Korean schema keys are retained below.

class MultinomialNBClassifier:
    """Multinomial Naive Bayes with Laplace smoothing."""
    def __init__(self, alpha: float = 0.75):
        self.alpha = alpha

    def fit(self, x, y, sample_weight=None):
        classes = np.unique(y)
        if sample_weight is None:
            sample_weight = np.ones_like(y, dtype=np.float32)
        feat = x.shape[1]
        log_prob = np.zeros((len(classes), feat), dtype=np.float64)
        log_prior = np.zeros(len(classes), dtype=np.float64)
        for k, c in enumerate(classes):
            mask = (y == c)
            w = sample_weight[mask]
            xc = x[mask]
            class_count = (xc * w[:, None]).sum(axis=0) + self.alpha
            log_prob[k] = np.log(class_count / class_count.sum())
            log_prior[k] = np.log(w.sum() + 1e-9)
        self.classes_ = classes
        self.log_prob_ = log_prob
        self.log_prior_ = log_prior - np.log(np.exp(log_prior).sum())
        return self

    def predict(self, x):
        scores = x @ self.log_prob_.T + self.log_prior_
        return self.classes_[np.argmax(scores, axis=1)]


class ComplementNBClassifier:
    """Complement Naive Bayes for imbalanced data."""
    def __init__(self, alpha: float = 0.75):
        self.alpha = alpha

    def fit(self, x, y, sample_weight=None):
        classes = np.unique(y)
        if sample_weight is None:
            sample_weight = np.ones_like(y, dtype=np.float32)
        feat = x.shape[1]
        weights = np.zeros((len(classes), feat), dtype=np.float64)
        for k, c in enumerate(classes):
            mask = (y != c)
            w = sample_weight[mask]
            xc = x[mask]
            comp_count = (xc * w[:, None]).sum(axis=0) + self.alpha
            log_p = np.log(comp_count / comp_count.sum())
            # Domain-specific processing step; Korean schema keys are retained below.
            weights[k] = -log_p / np.abs(log_p).sum()
        self.classes_ = classes
        self.weights_ = weights
        return self

    def predict(self, x):
        scores = x @ self.weights_.T
        return self.classes_[np.argmax(scores, axis=1)]


class LogOddsOVRClassifier:
    """Estimate class-specific log odds and combine them one-vs-rest."""
    def __init__(self, alpha: float = 0.75):
        self.alpha = alpha

    def fit(self, x, y, sample_weight=None):
        classes = np.unique(y)
        if sample_weight is None:
            sample_weight = np.ones_like(y, dtype=np.float32)
        feat = x.shape[1]
        weights = np.zeros((len(classes), feat), dtype=np.float64)
        biases = np.zeros(len(classes), dtype=np.float64)
        for k, c in enumerate(classes):
            pos_mask = (y == c)
            neg_mask = ~pos_mask
            pos_count = (x[pos_mask] * sample_weight[pos_mask, None]).sum(axis=0) + self.alpha
            neg_count = (x[neg_mask] * sample_weight[neg_mask, None]).sum(axis=0) + self.alpha
            pos_prob = pos_count / pos_count.sum()
            neg_prob = neg_count / neg_count.sum()
            weights[k] = np.log(pos_prob) - np.log(neg_prob)
            biases[k] = np.log(sample_weight[pos_mask].sum() + 1e-9) - \
                        np.log(sample_weight[neg_mask].sum() + 1e-9)
        self.classes_ = classes
        self.weights_ = weights
        self.biases_ = biases
        return self

    def predict(self, x):
        scores = x @ self.weights_.T + self.biases_
        return self.classes_[np.argmax(scores, axis=1)]


class RidgeOVRClassifier:
    """Closed-form L2-regularized one-vs-rest linear classifier."""
    def __init__(self, alpha: float = 2.0):
        self.alpha = alpha

    def fit(self, x, y, sample_weight=None):
        classes = np.unique(y)
        if sample_weight is None:
            sample_weight = np.ones_like(y, dtype=np.float32)
        # Domain-specific processing step; Korean schema keys are retained below.
        x_b = np.hstack([x, np.ones((x.shape[0], 1), dtype=x.dtype)])
        feat = x_b.shape[1]
        sw = sample_weight.astype(np.float64)
        xtwx = x_b.T @ (x_b * sw[:, None])
        reg = self.alpha * np.eye(feat)
        reg[-1, -1] = 0  # Domain-specific Korean-language processing rule.
        inv = np.linalg.pinv(xtwx + reg)
        coefs = []
        for c in classes:
            target = (y == c).astype(np.float64) * 2 - 1  # Domain-specific Korean-language processing rule.
            xtwy = x_b.T @ (target * sw)
            coef = inv @ xtwy
            coefs.append(coef)
        self.classes_ = classes
        self.coefs_ = np.vstack(coefs)
        return self

    def predict(self, x):
        x_b = np.hstack([x, np.ones((x.shape[0], 1), dtype=x.dtype)])
        scores = x_b @ self.coefs_.T
        return self.classes_[np.argmax(scores, axis=1)]


class NearestCentroidClassifier:
    """Classify observations by cosine similarity to class centroids."""
    def __init__(self, prior_strength: float = 0.05):
        self.prior_strength = prior_strength

    def fit(self, x, y, sample_weight=None):
        classes = np.unique(y)
        if sample_weight is None:
            sample_weight = np.ones_like(y, dtype=np.float32)
        sw = sample_weight.astype(np.float64)
        centroids = []
        priors = []
        for c in classes:
            mask = (y == c)
            w = sw[mask]
            xc = x[mask]
            centroid = (xc * w[:, None]).sum(axis=0) / max(w.sum(), 1e-9)
            n = np.linalg.norm(centroid)
            if n > 0:
                centroid = centroid / n
            centroids.append(centroid)
            priors.append(np.log(w.sum() + 1e-9))
        self.classes_ = classes
        self.centroids_ = np.vstack(centroids)
        self.log_prior_ = np.array(priors) - np.log(np.exp(priors).sum())
        return self

    def predict(self, x):
        # Domain-specific processing step; Korean schema keys are retained below.
        n = np.linalg.norm(x, axis=1, keepdims=True)
        n[n == 0] = 1.0
        x_n = x / n
        scores = x_n @ self.centroids_.T + self.prior_strength * self.log_prior_
        return self.classes_[np.argmax(scores, axis=1)]


# Domain-specific processing step; Korean schema keys are retained below.
ALGORITHM_FACTORIES = {
    'multinomial_nb':   lambda: MultinomialNBClassifier(alpha=0.75),
    'complement_nb':    lambda: ComplementNBClassifier(alpha=0.75),
    'log_odds_ovr':     lambda: LogOddsOVRClassifier(alpha=0.75),
    'ridge_ovr':        lambda: RidgeOVRClassifier(alpha=2.0),
    'nearest_centroid': lambda: NearestCentroidClassifier(prior_strength=0.05),
}

# Domain-specific processing step; Korean schema keys are retained below.
USE_TFIDF = {
    'multinomial_nb':   False,  # Domain-specific Korean-language processing rule.
    'complement_nb':    False,
    'log_odds_ovr':     False,
    'ridge_ovr':        True,  # Domain-specific Korean-language processing rule.
    'nearest_centroid': True,  # Domain-specific Korean-language processing rule.
}

print('Five classifier definitions complete')
print('Algorithms:', list(ALGORITHM_FACTORIES.keys()))

## 6.5 Load Training Data


In [ ]:
# Domain-specific processing step; Korean schema keys are retained below.
TARGETS = ['profanity_intensity', 'factual_assertion', 'sexual_expression', 'threat_level', 'repetition', 'identifiability', 'publicity']


def normalize_text(value) -> str:
    """Normalize whitespace and line breaks for text comparison."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return ''
    s = str(value).strip()
    return ' '.join(s.split())


# Domain-specific processing step; Korean schema keys are retained below.
MASTER_PATH = BASE / 'Risk_final' / 'comments_lawsuit_indexing_allv2.xlsx'

# Domain-specific processing step; Korean schema keys are retained below.
master = pd.read_excel(MASTER_PATH)
SOURCE_COLUMN_MAP = {
    '욕설 강도': 'profanity_intensity',
    '사실 적시성': 'factual_assertion',
    '성적 표현성': 'sexual_expression',
    '위협성': 'threat_level',
    '반복성': 'repetition',
    '특정성': 'identifiability',
    '공연성': 'publicity',
}
master = master.rename(columns=SOURCE_COLUMN_MAP)
print(f'Master file loaded: {len(master):,} rows')
print(f'Columns: {master.columns.tolist()}')

# Domain-specific processing step; Korean schema keys are retained below.
master['text_norm'] = master['text'].map(normalize_text)
master = master[master['text_norm'].str.len() > 0].copy()

# Domain-specific processing step; Korean schema keys are retained below.
for t in TARGETS:
    master[t] = pd.to_numeric(master[t], errors='coerce')

# Domain-specific processing step; Korean schema keys are retained below.
complete = master.dropna(subset=TARGETS).copy()
for t in TARGETS:
    complete[t] = complete[t].round().astype(int)

# Domain-specific processing step; Korean schema keys are retained below.
mask = np.ones(len(complete), dtype=bool)
for t in TARGETS:
    mask &= complete[t].between(1, 3).to_numpy()
complete = complete.loc[mask].copy()

# Domain-specific processing step; Korean schema keys are retained below.
dedup_rows = []
for _, grp in complete.groupby('text_norm', sort=False):
    row = grp.iloc[0].copy()
    if len(grp) > 1:
        for t in TARGETS:
            cnt = Counter(grp[t].astype(int).tolist())
            row[t] = sorted(cnt.items(), key=lambda x: (-x[1], x[0]))[0][0]
    dedup_rows.append(row)
training = pd.DataFrame(dedup_rows).reset_index(drop=True)

print()
print(f'Training data (after removing duplicates and missing values): {len(training):,} records')
print()
print('Class distribution by indicator:')
for t in TARGETS:
    dist = training[t].value_counts().sort_index().to_dict()
    print(f'  {t}: {dist}')


## 6.6 Five-Fold Cross-Validation

Stratified splitting preserves class proportions in every fold. This is important for highly imbalanced indicators such as threat and repetition, where an ordinary random split could leave a fold without minority-class observations.


In [ ]:
def make_stratified_folds(y: np.ndarray, n_splits: int = 5, seed: int = 42):
    """Create stratified folds by evenly distributing each class."""
    rng = np.random.default_rng(seed)
    folds = [[] for _ in range(n_splits)]
    for c in np.unique(y):
        idx = np.where(y == c)[0]
        rng.shuffle(idx)
        for i, v in enumerate(idx):
            folds[i % n_splits].append(int(v))
    result = []
    all_idx = np.arange(len(y))
    for valid_idx in folds:
        valid_idx = np.array(sorted(valid_idx), dtype=int)
        train_mask = np.ones(len(y), dtype=bool)
        train_mask[valid_idx] = False
        train_idx = all_idx[train_mask]
        result.append((train_idx, valid_idx))
    return result


def class_weight_vector(y: np.ndarray) -> np.ndarray:
    """Create inverse-frequency class weights to address imbalance."""
    counts = Counter(y.tolist())
    classes = sorted(counts)
    n = len(y)
    by_class = {c: n / (len(classes) * max(cnt, 1)) for c, cnt in counts.items()}
    weights = np.array([by_class[v] for v in y], dtype=np.float32)
    return weights / weights.mean()


def metrics_from_pred(y_true, y_pred, classes=(1, 2, 3)) -> dict:
    classes = list(classes)
    n = len(y_true)
    accuracy = float((y_true == y_pred).mean()) if n > 0 else 0.0
    mae = float(np.mean(np.abs(y_true - y_pred))) if n > 0 else 0.0
    f1_per_class = []
    support = []
    for c in classes:
        tp = int(((y_pred == c) & (y_true == c)).sum())
        fp = int(((y_pred == c) & (y_true != c)).sum())
        fn = int(((y_pred != c) & (y_true == c)).sum())
        sup = int((y_true == c).sum())
        prec = tp / (tp + fp) if (tp + fp) else 0.0
        rec = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) else 0.0
        f1_per_class.append(f1)
        support.append(sup)
    macro_f1 = float(np.mean(f1_per_class))
    weighted_f1 = float(sum(f * s for f, s in zip(f1_per_class, support)) /
                        max(sum(support), 1))
    return {'accuracy': accuracy, 'macro_f1': macro_f1,
            'weighted_f1': weighted_f1, 'mae': mae}


def cross_validate_one(texts, y, algorithm, n_splits=5, seed=42,
                       max_features=10000, min_df=1):
    """Run five-fold cross-validation for one algorithm and one indicator."""
    fold_metrics = []
    for fold_idx, (tr, va) in enumerate(make_stratified_folds(y, n_splits, seed), 1):
        train_texts = [texts[i] for i in tr]
        valid_texts = [texts[i] for i in va]
        vect = TokenVectorizer(max_features=max_features, min_df=min_df).fit(train_texts)
        if USE_TFIDF[algorithm]:
            x_train = vect.transform_tfidf(train_texts)
            x_valid = vect.transform_tfidf(valid_texts)
        else:
            x_train = vect.transform_counts(train_texts)
            x_valid = vect.transform_counts(valid_texts)
        y_tr = y[tr]
        y_va = y[va]
        sw = class_weight_vector(y_tr)
        model = ALGORITHM_FACTORIES[algorithm]()
        model.fit(x_train, y_tr, sample_weight=sw)
        y_pred = model.predict(x_valid)
        m = metrics_from_pred(y_va, y_pred)
        m['fold'] = fold_idx
        fold_metrics.append(m)

    avg = {
        'accuracy':    float(np.mean([m['accuracy']    for m in fold_metrics])),
        'macro_f1':    float(np.mean([m['macro_f1']    for m in fold_metrics])),
        'weighted_f1': float(np.mean([m['weighted_f1'] for m in fold_metrics])),
        'mae':         float(np.mean([m['mae']         for m in fold_metrics])),
        'folds':       len(fold_metrics),
    }
    return fold_metrics, avg


print('Cross-validation functions defined')

## 6.7 Evaluate All 35 Algorithm–Indicator Combinations

Five algorithms are evaluated against seven indicators using five-fold cross-validation.


In [ ]:
# Domain-specific processing step; Korean schema keys are retained below.
texts = training['text_norm'].tolist()
ALGORITHMS = list(ALGORITHM_FACTORIES.keys())

avg_rows = []
fold_rows = []

print('Starting algorithm-by-indicator evaluation...')
print('-' * 75)
for target in TARGETS:
    y = training[target].astype(int).to_numpy()
    for algo in ALGORITHMS:
        try:
            folds_m, avg_m = cross_validate_one(texts, y, algo)
        except Exception as e:
            print(f'  [SKIP] {target} × {algo}: {e}')
            continue
        avg_rows.append({
            'target': target, 'algorithm': algo,
            **{k: round(v, 4) for k, v in avg_m.items() if k != 'folds'},
            'folds': avg_m['folds'],
        })
        for fm in folds_m:
            fold_rows.append({
                'target': target, 'algorithm': algo, 'fold': fm['fold'],
                'accuracy': round(fm['accuracy'], 4),
                'macro_f1': round(fm['macro_f1'], 4),
                'weighted_f1': round(fm['weighted_f1'], 4),
                'mae': round(fm['mae'], 4),
            })
        print(f"  [OK] {target:10s} × {algo:18s} | "
              f"macro_f1={avg_m['macro_f1']:.4f}  acc={avg_m['accuracy']:.4f}  "
              f"mae={avg_m['mae']:.4f}")

avg_df = pd.DataFrame(avg_rows)
fold_df = pd.DataFrame(fold_rows)
print('-' * 75)
print(f'Total evaluation combinations: {len(avg_df)} combinations (= {len(TARGETS)} indicators × {len(ALGORITHMS)} Algorithms)')

## 6.8 Result Matrices — Macro F1 / Accuracy / MAE


In [ ]:
# Domain-specific processing step; Korean schema keys are retained below.
pivot_macro_f1 = avg_df.pivot(index='target', columns='algorithm', values='macro_f1')
pivot_acc      = avg_df.pivot(index='target', columns='algorithm', values='accuracy')
pivot_mae      = avg_df.pivot(index='target', columns='algorithm', values='mae')

# Domain-specific processing step; Korean schema keys are retained below.
pivot_macro_f1 = pivot_macro_f1.reindex(index=TARGETS, columns=ALGORITHMS)
pivot_acc      = pivot_acc.reindex(index=TARGETS, columns=ALGORITHMS)
pivot_mae      = pivot_mae.reindex(index=TARGETS, columns=ALGORITHMS)

print('=== Macro F1 matrix (rows=indicators, columns=algorithms) ===')
print(pivot_macro_f1.round(4).to_string())
print()
print('=== Accuracy matrix ===')
print(pivot_acc.round(4).to_string())
print()
print('=== MAE matrix (lower is better) ===')
print(pivot_mae.round(4).to_string())

In [ ]:
# Domain-specific processing step; Korean schema keys are retained below.
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, pivot, title, cmap, fmt in [
    (axes[0], pivot_macro_f1, 'Macro F1',  'YlOrRd', '.3f'),
    (axes[1], pivot_acc,      'Accuracy',   'YlGnBu', '.3f'),
    (axes[2], pivot_mae,      'MAE (lower is better)', 'YlOrBr_r', '.3f'),
]:
    im = ax.imshow(pivot.values, cmap=cmap, aspect='auto')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=30, ha='right', fontsize=9)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=10)
    ax.set_title(title, fontsize=12, fontweight='bold')
    # Domain-specific processing step; Korean schema keys are retained below.
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            v = pivot.values[i, j]
            ax.text(j, i, f'{v:{fmt}}', ha='center', va='center',
                    color='black', fontsize=9)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle('Five algorithms × seven indicators (mean five-fold CV performance)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Domain-specific processing step; Korean schema keys are retained below.
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()

colors = {
    'multinomial_nb':   '#4C72B0',
    'complement_nb':    '#DD8452',
    'log_odds_ovr':     '#55A467',
    'ridge_ovr':        '#C44E52',
    'nearest_centroid': '#8172B2',
}

for i, target in enumerate(TARGETS):
    ax = axes[i]
    sub = avg_df[avg_df['target'] == target].set_index('algorithm').reindex(ALGORITHMS)
    bars = ax.bar(sub.index, sub['macro_f1'],
                  color=[colors[a] for a in sub.index])
    # Domain-specific processing step; Korean schema keys are retained below.
    best_algo = sub['macro_f1'].idxmax()
    for b, a in zip(bars, sub.index):
        if a == best_algo:
            b.set_edgecolor('red')
            b.set_linewidth(2.5)
    for b, v in zip(bars, sub['macro_f1']):
        ax.annotate(f'{v:.3f}',
                    xy=(b.get_x() + b.get_width() / 2, b.get_height()),
                    xytext=(0, 3), textcoords='offset points',
                    ha='center', fontsize=9)
    ax.set_title(f'{target}', fontweight='bold')
    ax.set_ylabel('Macro F1')
    ax.set_ylim(0, max(sub['macro_f1']) * 1.25)
    ax.tick_params(axis='x', rotation=30)
    ax.grid(axis='y', linestyle=':', alpha=0.5)

axes[7].axis('off')  # Domain-specific Korean-language processing rule.
plt.suptitle('Macro F1 by indicator and algorithm (red border = best)',
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

## 6.9 Select and Interpret the Best Model for Each Indicator


In [ ]:
# Domain-specific processing step; Korean schema keys are retained below.
best_table = avg_df.sort_values(['target', 'macro_f1'], ascending=[True, False])\
                   .groupby('target', as_index=False).first()
best_table = best_table.set_index('target').reindex(TARGETS).reset_index()

print('=== Final model selected for each indicator (by Macro F1) ===')
print(best_table[['target', 'algorithm', 'macro_f1', 'accuracy',
                  'weighted_f1', 'mae']].to_string(index=False))
print()
print('=== Algorithm win counts ===')
print(best_table['algorithm'].value_counts().to_string())

In [ ]:
# Domain-specific processing step; Korean schema keys are retained below.
print('=' * 78)
print(f'{"All algorithm results by indicator (mean five-fold CV)":^78}')
print('=' * 78)

for target in TARGETS:
    sub = avg_df[avg_df['target'] == target].set_index('algorithm').reindex(ALGORITHMS)
    sorted_idx = sub.sort_values('macro_f1', ascending=False).index.tolist()
    best_algo = sorted_idx[0]

    # Domain-specific processing step; Korean schema keys are retained below.
    dist = training[target].value_counts().sort_index().to_dict()

    print()
    print(f'■ {target}  (class distribution: {dist})')
    print('  ' + '-' * 70)
    print(f'  {"Algorithms":<20}{"Macro F1":>10}{"Accuracy":>10}'
          f'{"Weighted F1":>14}{"MAE":>10}{"Rank":>6}')
    print('  ' + '-' * 70)
    for algo in ALGORITHMS:
        rank = sorted_idx.index(algo) + 1
        marker = ' ★' if algo == best_algo else '  '
        r = sub.loc[algo]
        print(f'  {algo:<20}'
              f'{r["macro_f1"]:>10.4f}'
              f'{r["accuracy"]:>10.4f}'
              f'{r["weighted_f1"]:>14.4f}'
              f'{r["mae"]:>10.4f}'
              f'{rank:>5}{marker}')

print()
print('=' * 78)

## 6.10 Train and Save Final Models

The best-performing algorithm for each indicator is trained on the full labeled dataset.


In [ ]:
# Domain-specific processing step; Korean schema keys are retained below.
import pickle

final_vectorizer = TokenVectorizer(max_features=10000, min_df=1).fit(texts)
counts_full  = final_vectorizer.transform_counts(texts)
tfidf_full   = final_vectorizer.transform_tfidf(texts)

target_models = {}
for target in TARGETS:
    best_row = best_table[best_table['target'] == target].iloc[0]
    algo = best_row['algorithm']
    y = training[target].astype(int).to_numpy()
    sw = class_weight_vector(y)
    model = ALGORITHM_FACTORIES[algo]()
    x_full = tfidf_full if USE_TFIDF[algo] else counts_full
    model.fit(x_full, y, sample_weight=sw)
    target_models[target] = {'algorithm': algo, 'model': model}
    print(f'  [LEARN] {target:10s} → {algo}')

bundle = {
    'targets': TARGETS,
    'vectorizer': final_vectorizer,
    'target_models': target_models,
    'use_tfidf_by_algo': USE_TFIDF,
    'training_rows': int(len(training)),
    'best_metrics': best_table.to_dict(orient='records'),
}

# Domain-specific processing step; Korean schema keys are retained below.
save_path = BASE / 'comment_scorer_model_final.pkl'
with open(save_path, 'wb') as f:
    pickle.dump(bundle, f)
print()
print(f'Final models saved: {save_path}')
print(f'Training sample size: {len(training):,} records')

In [ ]:
# Domain-specific processing step; Korean schema keys are retained below.
comments_path = BASE / 'comments_all.csv'
comments = load_csv_safe(comments_path)

print(f'All comments loaded: {len(comments):,} records')
print(comments.columns.tolist())

# Domain-specific processing step; Korean schema keys are retained below.
texts_all = comments['text'].fillna('').astype(str).tolist()

# Domain-specific processing step; Korean schema keys are retained below.
vect = bundle['vectorizer']
counts_all = vect.transform_counts(texts_all)
tfidf_all = vect.transform_tfidf(texts_all)

# Domain-specific processing step; Korean schema keys are retained below.
pred_cols = []

for target, info in bundle['target_models'].items():
    algo = info['algorithm']
    model = info['model']
    
    x_all = tfidf_all if bundle['use_tfidf_by_algo'][algo] else counts_all
    pred_col = f'pred_{target}'
    
    comments[pred_col] = model.predict(x_all).astype(int)
    pred_cols.append(pred_col)
    
    print(f'  [PREDICT] {target:10s} ← {algo}')

# Domain-specific processing step; Korean schema keys are retained below.
comments['pred_total_score'] = comments[pred_cols].sum(axis=1)

# Domain-specific processing step; Korean schema keys are retained below.
comments['pred_review_level'] = pd.cut(
    comments['pred_total_score'],
    bins=[0, 9, 14, 21],
    labels=['LOW', 'MEDIUM', 'HIGH'],
    include_lowest=True
)

# Domain-specific processing step; Korean schema keys are retained below.
output_path = RISK_DIR / 'comments_all_predicted_5algo.csv'
comments.to_csv(output_path, index=False, encoding='utf-8-sig')

print()
print(f'Prediction complete: {len(comments):,} records')
print(f'Output path: {output_path}')
print()
print(comments[['text'] + pred_cols + ['pred_total_score', 'pred_review_level']].head())


---
## 7. Combine Legal Elements and Estimate Offense Probabilities

### Seven indicators → four offenses

The seven predicted indicators are converted into four offense-level risk scores. Because criminal liability generally requires multiple legal elements to be satisfied jointly, the required elements are treated as AND conditions and combined multiplicatively. This conservative heuristic lowers the overall offense score whenever any required element receives a low score.

| Offense | Combination rule | Statutory reference |
|---|---|---|
| Insult | P(profanity intensity) × P(specificity) × P(publicity) | Criminal Act, Article 311 |
| Defamation | P(factual assertion) × P(specificity) × P(publicity) | Criminal Act, Article 307 |
| Threat | P(threat) × P(specificity) | Criminal Act, Article 283 |
| Obscene communication | P(sexual expression) × P(specificity) × P(publicity) | Act on Special Cases Concerning the Punishment of Sexual Crimes, Article 13 |

### Score-to-probability mapping

Predicted indicator scores are converted to risk weights. These values are not estimates of actual court outcomes; they are heuristic risk scores used for premium calculation.

| Score | Probability weight | Interpretation |
|---|---:|---|
| 1 | 0.10 | Low likelihood that the element is satisfied |
| 2 | 0.50 | Ambiguous; additional review is required |
| 3 | 0.90 | High likelihood that the element is satisfied |

For example, a comment with strong profanity still receives a limited insult-risk score if the target is unclear or publicity is low. The model therefore multiplies the three required elements instead of adding them.


In [ ]:
# Domain-specific processing step; Korean schema keys are retained below.
PROB_MAP = {1: 0.10, 2: 0.50, 3: 0.90}

def attach_offense_probabilities(df_scored):
    """
    Add legal-action probabilities by offense to a DataFrame containing seven indicator scores (1-3).
    """
    out = df_scored.copy()
    p = lambda col: out[col].map(PROB_MAP)

    out['p_insult']          = p('pred_profanity_intensity') * p('pred_identifiability') * p('pred_publicity')
    out['p_defamation']         = p('pred_factual_assertion') * p('pred_identifiability') * p('pred_publicity')
    out['p_threat']             = p('pred_threat_level') * p('pred_identifiability')
    out['p_obscene_communication']  = p('pred_sexual_expression') * p('pred_identifiability') * p('pred_publicity')
    out['p_max']             = out[['p_insult','p_defamation','p_threat','p_obscene_communication']].max(axis=1)

    return out

# Domain-specific processing step; Korean schema keys are retained below.
sim_data = []
for s1 in [1, 2, 3]:  # Domain-specific Korean-language processing rule.
    for s2 in [1, 2, 3]:  # Domain-specific Korean-language processing rule.
        for s3 in [1, 2, 3]:  # Domain-specific Korean-language processing rule.
            p = PROB_MAP[s1] * PROB_MAP[s2] * PROB_MAP[s3]
            sim_data.append({'profanity_intensity': s1, 'identifiability': s2, 'publicity': s3, 'insult_probability': round(p, 4)})

df_sim = pd.DataFrame(sim_data)
print('=== Example insult probability (profanity_intensity × identifiability × publicity) ===')
print(df_sim.pivot_table(
    index=['profanity_intensity','identifiability'], columns='publicity', values='insult_probability'
))

In [ ]:
# Domain-specific processing step; Korean schema keys are retained below.
pred_path = BASE / 'Risk_final/comments_all_predicted_5algo.csv'

try:
    scored_df = pd.read_csv(pred_path, encoding='utf-8-sig', low_memory=False)
except Exception:
    scored_df = load_csv_safe(pred_path)

print(f'Predicted comments loaded: {len(scored_df):,} records')
print(f'Columns: {scored_df.columns.tolist()}')

# -----------------------------
# Domain-specific processing step; Korean schema keys are retained below.
# -----------------------------
scored_df['like_count'] = pd.to_numeric(
    scored_df.get('like_count', 0),
    errors='coerce'
).fillna(0)

score_cols = [
    'pred_profanity_intensity',
    'pred_factual_assertion',
    'pred_sexual_expression',
    'pred_threat_level',
    'pred_repetition',
    'pred_identifiability',
    'pred_publicity',
]

for c in score_cols:
    scored_df[c] = pd.to_numeric(scored_df[c], errors='coerce').fillna(1).clip(1, 3).astype(int)

if 'pred_total_score' not in scored_df.columns:
    scored_df['pred_total_score'] = scored_df[score_cols].sum(axis=1)

if 'pred_review_level' not in scored_df.columns:
    scored_df['pred_review_level'] = pd.cut(
        scored_df['pred_total_score'],
        bins=[0, 9, 14, 21],
        labels=['LOW', 'MEDIUM', 'HIGH'],
        include_lowest=True
    )

# -----------------------------
# Domain-specific processing step; Korean schema keys are retained below.
# -----------------------------
SUBJECT_KEYWORDS = {
    'Subject_A': ['REDACTED_SUBJECT_A'],
    'Subject_B': ['REDACTED_SUBJECT_B'],
    'Subject_C': ['REDACTED_SUBJECT_C'],
}

def tag_person(title):
    t = str(title).lower()
    for person, kws in SUBJECT_KEYWORDS.items():
        if any(kw.lower() in t for kw in kws):
            return person
    return 'Other'

ISSUE_KW = {
    'controversy_or_scandal': ['논란', '사건', '갑질', '허위', '의혹', '논쟁', '비하', '짝퉁', 'redacted_event'],
    'news_report': ['뉴스', '보도', 'ytn', 'kbs', 'mbc', 'sbs', 'jtbc', '변호사'],
    'appearance_or_daily_life': ['몸매', '비키니', '셀카', '일상', 'vlog', '데일리', '예쁘', 'shorts'],
}

def tag_issue(title):
    t = str(title).lower()
    for issue, kws in ISSUE_KW.items():
        if any(kw.lower() in t for kw in kws):
            return issue
    return 'Other'

scored_df['person'] = scored_df['video_title'].map(tag_person)
scored_df['issue_type'] = scored_df['video_title'].map(tag_issue)

# -----------------------------
# Domain-specific processing step; Korean schema keys are retained below.
# -----------------------------
PROB_MAP = {1: 0.10, 2: 0.50, 3: 0.90}

def p(col):
    return scored_df[col].map(PROB_MAP)

scored_df['p_insult'] = (
    p('pred_profanity_intensity') *
    p('pred_identifiability') *
    p('pred_publicity')
)

scored_df['p_defamation'] = (
    p('pred_factual_assertion') *
    p('pred_identifiability') *
    p('pred_publicity')
)

scored_df['p_threat'] = (
    p('pred_threat_level') *
    p('pred_identifiability')
)

scored_df['p_obscene_communication'] = (
    p('pred_sexual_expression') *
    p('pred_identifiability') *
    p('pred_publicity')
)

# -----------------------------
# Domain-specific processing step; Korean schema keys are retained below.
# -----------------------------
LEGAL_COST = {
    'insult': 500_000,
    'defamation': 1_000_000,
    'threat': 1_000_000,
    'obscene_communication': 1_500_000,
}

LOADING_FACTOR = 1.4

incident_df = scored_df.groupby(
    ['video_id', 'video_title', 'person', 'issue_type'],
    as_index=False
).agg(
    comment_count=('text', 'size'),
    avg_total_score=('pred_total_score', 'mean'),
    max_total_score=('pred_total_score', 'max'),
    high_count=('pred_review_level', lambda s: int((s.astype(str) == 'HIGH').sum())),
    expected_insult=('p_insult', 'sum'),
    expected_defamation=('p_defamation', 'sum'),
    expected_threat=('p_threat', 'sum'),
    expected_obscene_communication=('p_obscene_communication', 'sum'),
    mean_p_insult=('p_insult', 'mean'),
    mean_p_defamation=('p_defamation', 'mean'),
    mean_p_threat=('p_threat', 'mean'),
    mean_p_obscene_communication=('p_obscene_communication', 'mean'),
)

incident_df['expected_legal_cost'] = (
    incident_df['expected_insult'] * LEGAL_COST['insult'] +
    incident_df['expected_defamation'] * LEGAL_COST['defamation'] +
    incident_df['expected_threat'] * LEGAL_COST['threat'] +
    incident_df['expected_obscene_communication'] * LEGAL_COST['obscene_communication']
)

incident_df['premium_pure'] = incident_df['expected_legal_cost']
incident_df['premium_gross'] = incident_df['expected_legal_cost'] * LOADING_FACTOR

incident_df = incident_df.sort_values(
    'expected_legal_cost',
    ascending=False
).reset_index(drop=True)

incident_df.insert(0, 'incident_rank', np.arange(1, len(incident_df) + 1))

# -----------------------------
# Domain-specific processing step; Korean schema keys are retained below.
# -----------------------------
person_df = incident_df.groupby('person', as_index=False).agg(
    incident_count=('video_id', 'size'),
    total_comments=('comment_count', 'sum'),
    avg_comments_per_incident=('comment_count', 'mean'),
    sum_expected_legal_cost=('expected_legal_cost', 'sum'),
    avg_expected_legal_cost_per_incident=('expected_legal_cost', 'mean'),
    avg_premium_gross=('premium_gross', 'mean'),
    max_premium_gross=('premium_gross', 'max'),
)

offense_df = pd.DataFrame({
    'offense': ['insult', 'defamation', 'threat', 'obscene_communication'],
    'expected_count': [
        scored_df['p_insult'].sum(),
        scored_df['p_defamation'].sum(),
        scored_df['p_threat'].sum(),
        scored_df['p_obscene_communication'].sum(),
    ],
    'legal_cost_per_case': [
        LEGAL_COST['insult'],
        LEGAL_COST['defamation'],
        LEGAL_COST['threat'],
        LEGAL_COST['obscene_communication'],
    ],
})

offense_df['expected_legal_cost'] = (
    offense_df['expected_count'] * offense_df['legal_cost_per_case']
)

issue_df = incident_df.groupby('issue_type', as_index=False).agg(
    incident_count=('video_id', 'size'),
    total_comments=('comment_count', 'sum'),
    sum_expected_legal_cost=('expected_legal_cost', 'sum'),
    avg_expected_legal_cost=('expected_legal_cost', 'mean'),
    avg_premium_gross=('premium_gross', 'mean'),
)

print(f'Incident detail: {len(incident_df):,} records')
print(f'Summary by subject: {len(person_df):,} records')
print(f'Summary by offense: {len(offense_df):,} records')
print(f'Summary by issue type: {len(issue_df):,} records')

incident_df.head()


In [ ]:
output_xlsx = RISK_DIR / 'incident_summary_from_5algov2.xlsx'

with pd.ExcelWriter(output_xlsx, engine='openpyxl') as writer:
    incident_df.to_excel(writer, sheet_name='incident_detail', index=False)
    person_df.to_excel(writer, sheet_name='subject_summary', index=False)
    offense_df.to_excel(writer, sheet_name='offense_summary', index=False)
    issue_df.to_excel(writer, sheet_name='issue_type_summary', index=False)

print(f'Save complete: {output_xlsx}')

In [ ]:
# Domain-specific processing step; Korean schema keys are retained below.

offense_prob_df = pd.DataFrame({
    'offense': ['insult', 'defamation', 'threat', 'obscene_communication'],
    'average_legal_action_probability': [
        scored_df['p_insult'].mean(),
        scored_df['p_defamation'].mean(),
        scored_df['p_threat'].mean(),
        scored_df['p_obscene_communication'].mean(),
    ],
    'expected_qualifying_count': [
        scored_df['p_insult'].sum(),
        scored_df['p_defamation'].sum(),
        scored_df['p_threat'].sum(),
        scored_df['p_obscene_communication'].sum(),
    ],
})

fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.bar(
    offense_prob_df['offense'],
    offense_prob_df['average_legal_action_probability'],
    color=['#4C72B0', '#DD8452', '#55A467', '#C44E52']
)

for b, v in zip(bars, offense_prob_df['average_legal_action_probability']):
    ax.text(
        b.get_x() + b.get_width() / 2,
        b.get_height() + 0.002,
        f'{v:.4f}',
        ha='center',
        fontsize=10
    )

ax.set_title('Average complaint-risk probability by offense')
ax.set_ylabel('Average probability')

max_v = offense_prob_df['average_legal_action_probability'].max()
ax.set_ylim(0, max_v * 1.3 if max_v > 0 else 0.1)

plt.tight_layout()
plt.show()

print(offense_prob_df)


---
## 8. Aggregate comments into Video-Level incidents

**Incident definition:** one `video_id` equals one incident. comments posted under the same video share a common publication context and are aggregated as a single incident. The expected number of claims per incident is the sum of comment-level offense probabilities and corresponds to claim frequency λ.


In [ ]:
# Domain-specific processing step; Korean schema keys are retained below.

required_vars = ['incident_df', 'person_df']
missing_vars = [v for v in required_vars if v not in globals()]

if missing_vars:
    print(f'Required DataFrames are missing: {missing_vars}')
    print('Run the incident aggregation cell based on comments_all_predicted_5algo.csv first.')
else:
    print('=== Incident-level statistics ===')
    print(f'Total incidents: {len(incident_df):,} records')
    print(f'Total comments: {incident_df["comment_count"].sum():,} records')
    print(f'Total expected legal cost: {incident_df["expected_legal_cost"].sum():,.0f} KRW')
    print(f'total gross premium: {incident_df["premium_gross"].sum():,.0f} KRW')
    print()

    # Domain-specific processing step; Korean schema keys are retained below.
    print('=== Summary by subject ===')

    display_cols = [
        'person',
        'incident_count',
        'total_comments',
        'sum_expected_legal_cost',
        'avg_expected_legal_cost_per_incident',
        'avg_premium_gross',
        'max_premium_gross',
    ]

    display_cols = [c for c in display_cols if c in person_df.columns]

    person_display = person_df[display_cols].copy()

    money_cols = [
        'sum_expected_legal_cost',
        'avg_expected_legal_cost_per_incident',
        'avg_premium_gross',
        'max_premium_gross',
    ]

    for c in money_cols:
        if c in person_display.columns:
            person_display[c] = person_display[c].map(lambda x: f'{x:,.0f} KRW')

    if 'avg_comments_per_incident' in person_display.columns:
        person_display['avg_comments_per_incident'] = person_display['avg_comments_per_incident'].map(lambda x: f'{x:,.1f}')

    print(person_display.to_string(index=False))
    print()

    # Domain-specific processing step; Korean schema keys are retained below.
    print('=== Top 10 incidents by premium ===')

    show_cols = [
        'incident_rank',
        'person',
        'issue_type',
        'comment_count',
        'avg_total_score',
        'high_count',
        'expected_legal_cost',
        'premium_gross',
        'video_title',
    ]

    show_cols = [c for c in show_cols if c in incident_df.columns]

    top10 = incident_df[show_cols].head(10).copy()

    for c in ['expected_legal_cost', 'premium_gross']:
        if c in top10.columns:
            top10[c] = top10[c].map(lambda x: f'{x:,.0f} KRW')

    if 'avg_total_score' in top10.columns:
        top10['avg_total_score'] = top10['avg_total_score'].map(lambda x: f'{x:.2f}')

    print(top10.to_string(index=False))


In [ ]:
# Domain-specific processing step; Korean schema keys are retained below.

if 'incident_df' not in globals():
    print('incident_df is unavailable. Run the incident aggregation cell based on comments_all_predicted_5algo.csv first.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # -----------------------------
    # Domain-specific processing step; Korean schema keys are retained below.
    # -----------------------------
    offense_cols_expected = [
        'expected_insult',
        'expected_defamation',
        'expected_threat',
        'expected_obscene_communication',
    ]

    offense_cols_present = [
        c for c in offense_cols_expected
        if c in incident_df.columns
    ]

    if offense_cols_present and 'person' in incident_df.columns:
        grp = incident_df.groupby('person')[offense_cols_present].sum()

        # Domain-specific processing step; Korean schema keys are retained below.
        row_sum = grp.sum(axis=1).replace(0, np.nan)
        grp_pct = grp.div(row_sum, axis=0).fillna(0) * 100

        label_map = {
            'expected_insult': 'insult',
            'expected_defamation': 'defamation',
            'expected_threat': 'threat',
            'expected_obscene_communication': 'obscene_communication',
        }

        grp_pct = grp_pct.rename(columns=label_map)

        grp_pct.plot(
            kind='bar',
            stacked=True,
            ax=axes[0],
            color=['#4C72B0', '#DD8452', '#55A467', '#C44E52']
        )

        axes[0].set_title('Offense composition by subject (%)')
        axes[0].set_xlabel('Subject')
        axes[0].set_ylabel('Share (%)')
        axes[0].tick_params(axis='x', rotation=0)
        axes[0].set_ylim(0, 100)
        axes[0].legend(title='offense', bbox_to_anchor=(1.02, 1.0), loc='upper left')
    else:
        axes[0].text(
            0.5, 0.5,
            'Expected-count columns by offense are missing.',
            ha='center', va='center',
            transform=axes[0].transAxes
        )
        axes[0].set_axis_off()

    # -----------------------------
    # Domain-specific processing step; Korean schema keys are retained below.
    # -----------------------------
    needed_cols = {'comment_count', 'premium_gross', 'person'}

    if needed_cols.issubset(incident_df.columns):
        colors_map = {
            'Subject_A': '#4C72B0',
            'Subject_B': '#C44E52',
            'Subject_C': '#DD8452',
            'Other': '#55A467',
        }

        for person, grp_data in incident_df.groupby('person'):
            axes[1].scatter(
                grp_data['comment_count'],
                grp_data['premium_gross'] / 1e6,
                alpha=0.55,
                label=person,
                color=colors_map.get(person, 'gray'),
                s=35
            )

        axes[1].set_title('incident-level comments vs gross premium')
        axes[1].set_xlabel('comments')
        axes[1].set_ylabel('gross premium (KRW millions)')
        axes[1].legend(title='Subject')
        axes[1].grid(True, alpha=0.3)
    else:
        axes[1].text(
            0.5, 0.5,
            'comment_count, premium_gross, person columns are required.',
            ha='center', va='center',
            transform=axes[1].transAxes
        )
        axes[1].set_axis_off()

    plt.tight_layout()
    plt.show()


In [ ]:
# Domain-specific processing step; Korean schema keys are retained below.

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# -----------------------------
# Domain-specific processing step; Korean schema keys are retained below.
# -----------------------------
person_cost = (
    incident_df
    .groupby('person', as_index=False)
    .agg(
        incident_count=('video_id', 'size'),
        total_comments=('comment_count', 'sum'),
        total_premium_gross=('premium_gross', 'sum'),
        avg_premium_gross=('premium_gross', 'mean')
    )
    .sort_values('total_premium_gross', ascending=False)
)

bars = axes[0].bar(
    person_cost['person'],
    person_cost['total_premium_gross'] / 1e8,
    color=['#C44E52', '#DD8452', '#55A467', '#4C72B0'][:len(person_cost)]
)

for b, v in zip(bars, person_cost['total_premium_gross'] / 1e8):
    axes[0].text(
        b.get_x() + b.get_width() / 2,
        b.get_height(),
        f'{v:.1f}100M',
        ha='center',
        va='bottom',
        fontsize=10
    )

axes[0].set_title('by subject total gross premium')
axes[0].set_xlabel('Subject')
axes[0].set_ylabel('Total gross premium (KRW 100 million)')
axes[0].grid(axis='y', alpha=0.3)

# -----------------------------
# Domain-specific processing step; Korean schema keys are retained below.
# Domain-specific processing step; Korean schema keys are retained below.
# -----------------------------
colors_map = {
    'Subject_A': '#4C72B0',
    'Subject_B': '#C44E52',
    'Subject_C': '#DD8452',
    'Other': '#55A467',
}

for person, g in incident_df.groupby('person'):
    axes[1].scatter(
        g['comment_count'],
        g['premium_gross'] / 1e6,
        s=np.clip(g['high_count'] * 8 + 20, 20, 220),
        alpha=0.55,
        label=person,
        color=colors_map.get(person, 'gray'),
        edgecolor='white',
        linewidth=0.5
    )

axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].set_title('incident-level comments vs gross premium')
axes[1].set_xlabel('Comments (log scale)')
axes[1].set_ylabel('Gross premium (KRW millions, log scale)')
axes[1].legend(title='Subject')
axes[1].grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

person_cost


At a 1% conversion rate, the estimated amounts are approximately KRW 428 million and KRW 319 million. The KRW 42.8 billion figure should be interpreted as risk exposure rather than the expected converted amount.

---
## 9. premium Calculation and Visualization

An incident is defined by `video_id`; comment-level benefits are not accumulated within the same video. A video is treated as a coverage-review candidate only when its average risk score meets the threshold. Each covered video receives a fixed benefit.

$$Covered_i = 1[AvgRisk_i \ge 0.8000]$$

$$Grosspremium_i = 1.4 	imes Covered_i 	imes 1{,}000{,}000$$

| Component | Setting |
|---|---|
| Incident unit | One `video_id` |
| Video risk score | Mean comment-level risk score within the video |
| Coverage threshold | Average risk score ≥ 0.8000 |
| Coverage design | Fixed benefit per candidate video |
| Fixed benefit limit | KRW 1,000,000 |
| Comment-level accumulation | None within the same video |
| Loading factor | 1.4, including safety loading and expenses |


In [ ]:
# Domain-specific processing step; Korean schema keys are retained below.

pred_path = BASE / 'Risk_final/comments_all_predicted_5algo.csv'
scored_df = pd.read_csv(pred_path, encoding='utf-8-sig', low_memory=False)

print(f'Entire Estimate comment Load: {len(scored_df):,} records')
print(scored_df.columns.tolist())

# Domain-specific processing step; Korean schema keys are retained below.
score_cols = [
    'pred_profanity_intensity',
    'pred_factual_assertion',
    'pred_sexual_expression',
    'pred_threat_level',
    'pred_repetition',
    'pred_identifiability',
    'pred_publicity',
]

for c in score_cols:
    scored_df[c] = pd.to_numeric(scored_df[c], errors='coerce').fillna(1).clip(1, 3).astype(int)

if 'pred_total_score' not in scored_df.columns:
    scored_df['pred_total_score'] = scored_df[score_cols].sum(axis=1)

# Domain-specific processing step; Korean schema keys are retained below.
SUBJECT_KEYWORDS = {
    'Subject_A': ['REDACTED_SUBJECT_A'],
    'Subject_B': ['REDACTED_SUBJECT_B'],
    'Subject_C': ['REDACTED_SUBJECT_C'],
}

def tag_person(title):
    t = str(title).lower()
    for person, kws in SUBJECT_KEYWORDS.items():
        if any(kw.lower() in t for kw in kws):
            return person
    return 'Other'

scored_df['person'] = scored_df['video_title'].map(tag_person)

# Domain-specific processing step; Korean schema keys are retained below.
PROB_MAP = {1: 0.10, 2: 0.50, 3: 0.90}

def prob(col):
    return scored_df[col].map(PROB_MAP)

scored_df['p_insult'] = (
    prob('pred_profanity_intensity') *
    prob('pred_identifiability') *
    prob('pred_publicity')
)

scored_df['p_defamation'] = (
    prob('pred_factual_assertion') *
    prob('pred_identifiability') *
    prob('pred_publicity')
)

scored_df['p_threat'] = (
    prob('pred_threat_level') *
    prob('pred_identifiability')
)

scored_df['p_obscene_communication'] = (
    prob('pred_sexual_expression') *
    prob('pred_identifiability') *
    prob('pred_publicity')
)

print('Offense probability columns created')
print(scored_df[['p_insult', 'p_defamation', 'p_threat', 'p_obscene_communication']].head())


In [ ]:
print(f'scored_df row count: {len(scored_df):,}')
print(scored_df.columns.tolist())

In [ ]:
# Domain-specific processing step; Korean schema keys are retained below.

RISK_THRESHOLD = 0.8000
FIXED_BENEFIT_PER_VIDEO = 1_000_000
LOADING_FACTOR = 1.4

# Domain-specific processing step; Korean schema keys are retained below.
# Domain-specific processing step; Korean schema keys are retained below.
scored_df['comment_risk_score'] = 1 - (
    (1 - scored_df['p_insult']) * 
    (1 - scored_df['p_defamation']) * 
    (1 - scored_df['p_threat']) * 
    (1 - scored_df['p_obscene_communication'])
)


# Domain-specific processing step; Korean schema keys are retained below.
incident_flat_df = (
    scored_df
    .groupby(['video_id', 'video_title', 'person'], as_index=False)
    .agg(
        comment_count=('text', 'size'),
        avg_risk_score=('comment_risk_score', 'mean'),
        max_risk_score=('comment_risk_score', 'max'),
        avg_total_score=('pred_total_score', 'mean'),
    )
)

# Domain-specific processing step; Korean schema keys are retained below.
incident_flat_df['is_covered'] = (
    incident_flat_df['avg_risk_score'] >= RISK_THRESHOLD
)

# Domain-specific processing step; Korean schema keys are retained below.
incident_flat_df['fixed_benefit'] = np.where(
    incident_flat_df['is_covered'],
    FIXED_BENEFIT_PER_VIDEO,
    0
)

# Domain-specific processing step; Korean schema keys are retained below.
incident_flat_df['gross_premium'] = (
    incident_flat_df['fixed_benefit'] * LOADING_FACTOR
)

# Domain-specific processing step; Korean schema keys are retained below.
incident_flat_df = incident_flat_df.sort_values(
    ['is_covered', 'avg_risk_score'],
    ascending=[False, False]
).reset_index(drop=True)

incident_flat_df



In [ ]:
print(f'All comments: {len(scored_df):,} records')
print(f'All incidents (video_id): {len(incident_flat_df):,} records')
print(f'coverage candidate incidents: {incident_flat_df["is_covered"].sum():,} records')
print(f'Coverage-candidate ratio: {incident_flat_df["is_covered"].mean():.1%}')

covered_comments = incident_flat_df.loc[
    incident_flat_df['is_covered'],
    'comment_count'
].sum()

print(f'Comments in coverage-candidate incidents: {covered_comments:,} records')
print(f'Average comments in coverage-candidate incidents: {covered_comments / incident_flat_df["is_covered"].sum():,.1f} records')


In [ ]:
# Domain-specific processing step; Korean schema keys are retained below.

RISK_THRESHOLD = 0.8000

if 'incident_flat_df' not in globals():
    print('incident_flat_df is unavailable. Run the baseline scenario cell first.')
else:
    risk = incident_flat_df['avg_risk_score'].dropna()

    total_n = len(risk)
    covered_n = (risk >= RISK_THRESHOLD).sum()
    excluded_n = total_n - covered_n
    covered_ratio = covered_n / total_n

    fig, ax = plt.subplots(figsize=(10, 6))

    counts, bins, patches = ax.hist(
        risk,
        bins=40,
        range=(0, 1),
        color='#3F4A5A',
        alpha=0.9,
        edgecolor='white',
        linewidth=0.6
    )

    # Domain-specific processing step; Korean schema keys are retained below.
    ax.axvline(
        RISK_THRESHOLD,
        color='red',
        linewidth=2.2
    )

    ymax = counts.max()

    ax.text(
        RISK_THRESHOLD + 0.01,
        ymax * 0.92,
        f'threshold {RISK_THRESHOLD:.2f}',
        color='red',
        fontsize=11,
        fontweight='bold'
    )

    # Domain-specific processing step; Korean schema keys are retained below.
    ax.axvspan(0, RISK_THRESHOLD, color='#4C72B0', alpha=0.05)
    ax.axvspan(RISK_THRESHOLD, 1, color='#C44E52', alpha=0.08)

    # Domain-specific processing step; Korean schema keys are retained below.
    ax.text(
        0.02,
        ymax * 0.86,
        f'non-coverage candidate: {excluded_n:,} records ({1-covered_ratio:.1%})',
        fontsize=11,
        color='#2F3A4A'
    )

    ax.text(
        RISK_THRESHOLD + 0.03,
        ymax * 0.78,
        f'Coverage candidate: {covered_n:,} records ({covered_ratio:.1%})',
        fontsize=11,
        color='#C44E52',
        fontweight='bold'
    )

    ax.set_title(f'Video-level average risk score distribution\nthreshold {RISK_THRESHOLD:.2f} threshold')
    ax.set_xlabel('Video-level average risk score')
    ax.set_ylabel('incidents')

    ax.set_xlim(0, 1)
    ax.set_ylim(0, ymax * 1.12)
    ax.grid(axis='y', alpha=0.25)

    plt.tight_layout()
    plt.show()

    print(f'All incidents: {total_n:,} records')
    print(f'Coverage candidate: {covered_n:,} records ({covered_ratio:.1%})')
    print(f'non-coverage candidate: {excluded_n:,} records ({1-covered_ratio:.1%})')


In [ ]:
# Domain-specific processing step; Korean schema keys are retained below.

thresholds = [0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]

threshold_check = pd.DataFrame({
    'threshold': thresholds,
    'covered_incidents': [(incident_flat_df['avg_risk_score'] >= t).sum() for t in thresholds],
})

threshold_check['covered_ratio'] = (
    threshold_check['covered_incidents'] / len(incident_flat_df)
)

threshold_check


In [ ]:
# Domain-specific processing step; Korean schema keys are retained below.

required_vars = ['scored_df', 'incident_flat_df']
missing_vars = [v for v in required_vars if v not in globals()]

if missing_vars:
    print(f'Required DataFrames are missing: {missing_vars}')
    print('Load all prediction data and run the baseline scenario cell first.')
else:
    premium_values = incident_flat_df['gross_premium']

    print('[Key calculation results: baseline scenario]')
    print(f'Analyzed comments      : {len(scored_df):,} records')
    print(f'Total incidents        : {len(incident_flat_df):,} records')
    print(f'coverage candidate incidents : {incident_flat_df["is_covered"].sum():,} records')
    print(f'Coverage-candidate ratio    : {incident_flat_df["is_covered"].mean():.1%}')
    print()

    print('[Calculation assumptions]')
    print('  Incident unit        : video_id')
    print(f'  Average-risk threshold: {RISK_THRESHOLD:.4f} or higher')
    print(f'  Fixed benefit limit    : {FIXED_BENEFIT_PER_VIDEO:,} KRW / video')
    print(f'  Loading factor        : {LOADING_FACTOR:.1f}')
    print(f'  Maximum gross premium per video: {FIXED_BENEFIT_PER_VIDEO * LOADING_FACTOR:,.0f} KRW')
    print()

    print('[Gross premium distribution per incident]')
    print(f'  Mean  : {premium_values.mean():>15,.0f} KRW')
    print(f'  Median : {premium_values.median():>15,.0f} KRW')
    print(f'  P90   : {premium_values.quantile(0.90):>15,.0f} KRW')
    print(f'  P99   : {premium_values.quantile(0.99):>15,.0f} KRW')
    print(f'  Maximum: {premium_values.max():>15,.0f} KRW')
    print()

    print(f'Total fixed benefit     : {incident_flat_df["fixed_benefit"].sum():,} KRW')
    print(f'total gross premium      : {incident_flat_df["gross_premium"].sum():,} KRW')


In [ ]:
# Domain-specific processing step; Korean schema keys are retained below.

if 'incident_flat_df' not in globals():
    print('incident_flat_df is unavailable. Run the baseline scenario cell first.')
else:
    by_person = (
        incident_flat_df
        .groupby('person', as_index=False)
        .agg(
            incident_count=('video_id', 'size'),
            covered_count=('is_covered', 'sum'),
            total_comments=('comment_count', 'sum'),
            avg_risk_score=('avg_risk_score', 'mean'),
            avg_premium_gross=('gross_premium', 'mean'),
            total_premium_gross=('gross_premium', 'sum'),
        )
        .sort_values('total_premium_gross', ascending=False)
    )

    print('=== Baseline premium by subject ===')
    print(by_person.to_string(index=False))

    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.bar(
        by_person['person'],
        by_person['total_premium_gross'] / 1e6,
        color=['#4C72B0', '#DD8452', '#55A467', '#C44E52'][:len(by_person)]
    )
    ax.set_title('Total gross premium by subject (baseline scenario)')
    ax.set_ylabel('gross premium (KRW millions)')
    for b, v in zip(bars, by_person['total_premium_gross']):
        ax.text(
            b.get_x() + b.get_width() / 2,
            b.get_height(),
            f'{v/1e6:.1f}M',
            ha='center',
            va='bottom',
            fontsize=10
        )
    plt.tight_layout()
    plt.show()


In [ ]:
# Domain-specific processing step; Korean schema keys are retained below.
# Domain-specific processing step; Korean schema keys are retained below.
# Domain-specific processing step; Korean schema keys are retained below.

if 'incident_flat_df' not in globals():
    print('incident_flat_df is unavailable. Run the baseline scenario cell first.')
else:
    coverage_counts = (
        incident_flat_df['is_covered']
        .value_counts()
        .rename(index={False: 'non-coverage candidate', True: 'Coverage candidate'})
        .reindex(['non-coverage candidate', 'Coverage candidate'])
        .fillna(0)
        .astype(int)
    )

    fig, ax = plt.subplots(figsize=(7, 5))
    bars = ax.bar(
        coverage_counts.index.astype(str),
        coverage_counts.values,
        color=['#B0B0B0', '#C44E52']
    )

    for b, v in zip(bars, coverage_counts.values):
        ax.text(
            b.get_x() + b.get_width() / 2,
            b.get_height(),
            f'{v:,} records',
            ha='center',
            va='bottom',
            fontsize=11
        )

    ax.set_title('Baseline scenario: Coverage candidate incidents')
    ax.set_xlabel('category')
    ax.set_ylabel('incidents')
    plt.tight_layout()
    plt.show()


In [ ]:
# Domain-specific processing step; Korean schema keys are retained below.

if 'incident_flat_df' not in globals():
    print('incident_flat_df is unavailable. Run the baseline scenario cell first.')
else:
    by_person = (
        incident_flat_df
        .groupby('person', as_index=False)
        .agg(
            incident_count=('video_id', 'size'),
            covered_count=('is_covered', 'sum'),
            avg_premium_gross=('gross_premium', 'mean'),
            total_premium_gross=('gross_premium', 'sum')
        )
        .sort_values('avg_premium_gross', ascending=True)
    )

    fig, ax = plt.subplots(figsize=(8, 5))

    ax.barh(
        by_person['person'],
        by_person['avg_premium_gross'] / 1e6,
        color='#55A467'
    )

    for i, v in enumerate(by_person['avg_premium_gross'] / 1e6):
        ax.text(
            v,
            i,
            f'{v:.2f}M',
            va='center',
            ha='left',
            fontsize=9
        )

    ax.set_title('Average gross premium by subject')
    ax.set_xlabel('gross premium (KRW millions)')

    plt.tight_layout()
    plt.show()

    display(by_person)


In [ ]:
# Domain-specific processing step; Korean schema keys are retained below.

required_cols = ['p_insult', 'p_defamation', 'p_threat', 'p_obscene_communication']
missing_cols = [c for c in required_cols if 'scored_df' not in globals() or c not in scored_df.columns]

if missing_cols:
    print(f'Required probability columns are missing: {missing_cols}')
    print('Load all prediction data and run the offense-probability cell first.')
else:
    offense_prob_df = pd.DataFrame({
        'offense': ['insult', 'defamation', 'threat', 'obscene_communication'],
        'average_legal_action_probability': [
            scored_df['p_insult'].mean(),
            scored_df['p_defamation'].mean(),
            scored_df['p_threat'].mean(),
            scored_df['p_obscene_communication'].mean(),
        ],
    })

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(
        offense_prob_df['offense'],
        offense_prob_df['average_legal_action_probability'],
        color=['#4C72B0', '#DD8452', '#55A467', '#C44E52']
    )
    for b, v in zip(bars, offense_prob_df['average_legal_action_probability']):
        ax.text(
            b.get_x() + b.get_width() / 2,
            b.get_height() + 0.002,
            f'{v:.4f}',
            ha='center',
            fontsize=10
        )
    ax.set_title('Average complaint-risk probability by offense\n(based on all predicted comments)')
    ax.set_ylabel('Average probability')
    max_v = offense_prob_df['average_legal_action_probability'].max()
    ax.set_ylim(0, max_v * 1.3 if max_v > 0 else 0.1)
    plt.tight_layout()
    plt.show()

    display(offense_prob_df)


In [ ]:
# ============================================================
# Domain-specific processing step; Korean schema keys are retained below.
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Domain-specific processing step; Korean schema keys are retained below.
# incident_flat_df
# FIXED_BENEFIT_PER_VIDEO
# LOADING_FACTOR

premium_summary = pd.DataFrame({
    'item': [
        'Average risk premium per video',
        'Average gross premium per video',
        'Maximum benefit per video',
        'total risk premium',
        'total gross premium',
        'maximum_total_benefit',
    ],
    'amount': [
        incident_flat_df['fixed_benefit'].mean(),
        incident_flat_df['gross_premium'].mean(),
        FIXED_BENEFIT_PER_VIDEO,
        incident_flat_df['fixed_benefit'].sum(),
        incident_flat_df['gross_premium'].sum(),
        FIXED_BENEFIT_PER_VIDEO * len(incident_flat_df),
    ]
})

premium_summary['formatted_amount'] = premium_summary['amount'].map(lambda x: f'{x:,.0f} KRW')

display(premium_summary[['item', 'formatted_amount']])

print('=== Per-video premium summary ===')
print(f'Video count: {len(incident_flat_df):,} records')
print(f'Average risk premium per video: {incident_flat_df["fixed_benefit"].mean():,.0f} KRW')
print(f'Average gross premium per video: {incident_flat_df["gross_premium"].mean():,.0f} KRW')
print(f'Maximum benefit per video: {FIXED_BENEFIT_PER_VIDEO:,.0f} KRW')

# -----------------------------
# Domain-specific processing step; Korean schema keys are retained below.
# -----------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Domain-specific processing step; Korean schema keys are retained below.
plot_df = premium_summary.iloc[:3].copy()

bars = axes[0].bar(
    plot_df['item'],
    plot_df['amount'] / 1e6,
    color=['#4C72B0', '#DD8452', '#C44E52']
)

for b, v in zip(bars, plot_df['amount']):
    axes[0].text(
        b.get_x() + b.get_width() / 2,
        b.get_height(),
        f'{v/1e6:.2f}M',
        ha='center',
        va='bottom',
        fontsize=10
    )

axes[0].set_title('Per-video premium summary')
axes[0].set_ylabel('amount (KRW millions)')
axes[0].tick_params(axis='x', rotation=15)
axes[0].grid(axis='y', alpha=0.3)

# Domain-specific processing step; Korean schema keys are retained below.
axes[1].hist(
    incident_flat_df['gross_premium'] / 1e6,
    bins=20,
    color='#55A467',
    edgecolor='white',
    alpha=0.9
)

axes[1].set_title('by video gross premium distribution')
axes[1].set_xlabel('gross premium (KRW millions)')
axes[1].set_ylabel('video_count')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Domain-specific processing step; Korean schema keys are retained below.
# ============================================================

person_video_premium = (
    incident_flat_df
    .groupby('person', as_index=False)
    .agg(
        video_count=('video_id', 'size'),
        coverage_candidate_video_count=('is_covered', 'sum'),
        total_comment_count=('comment_count', 'sum'),
        average_risk_score=('avg_risk_score', 'mean'),
        risk_premium_per_video=('fixed_benefit', 'mean'),
        gross_premium_per_video=('gross_premium', 'mean'),
        total_risk_premium=('fixed_benefit', 'sum'),
        total_gross_premium=('gross_premium', 'sum'),
    )
)

person_video_premium['coverage_candidate_ratio'] = (
    person_video_premium['coverage_candidate_video_count'] /
    person_video_premium['video_count']
)

person_video_premium['maximum_benefit_per_video'] = FIXED_BENEFIT_PER_VIDEO

person_video_premium = person_video_premium.sort_values(
    'gross_premium_per_video',
    ascending=False
)

display_df = person_video_premium.copy()

for c in [
    'risk_premium_per_video',
    'gross_premium_per_video',
    'total_risk_premium',
    'total_gross_premium',
    'maximum_benefit_per_video',
]:
    display_df[c] = display_df[c].map(lambda x: f'{x:,.0f} KRW')

display_df['average_risk_score'] = display_df['average_risk_score'].map(lambda x: f'{x:.3f}')
display_df['coverage_candidate_ratio'] = display_df['coverage_candidate_ratio'].map(lambda x: f'{x:.1%}')

display(display_df)


In [ ]:
# ============================================================
# Domain-specific processing step; Korean schema keys are retained below.
# ============================================================

fig, ax = plt.subplots(figsize=(9, 5))

x = np.arange(len(person_video_premium))
width = 0.35

ax.bar(
    x - width / 2,
    person_video_premium['risk_premium_per_video'] / 1e6,
    width,
    label='Risk premium per video',
    color='#4C72B0'
)

ax.bar(
    x + width / 2,
    person_video_premium['gross_premium_per_video'] / 1e6,
    width,
    label='Gross premium per video',
    color='#C44E52'
)

ax.axhline(
    FIXED_BENEFIT_PER_VIDEO / 1e6,
    color='gray',
    linestyle='--',
    linewidth=1.5,
    label='Maximum benefit per video'
)

ax.set_title('Premium per video by subject')
ax.set_xticks(x)
ax.set_xticklabels(person_video_premium['person'])
ax.set_ylabel('amount (KRW millions)')
ax.legend()
ax.grid(axis='y', alpha=0.3)

for i, v in enumerate(person_video_premium['gross_premium_per_video'] / 1e6):
    ax.text(
        i + width / 2,
        v,
        f'{v:.2f}M',
        ha='center',
        va='bottom',
        fontsize=9
    )

plt.tight_layout()
plt.show()


---
## 10. Summary and Limitations

### Key results

The following results use the revised score-to-probability mapping (1→0.20, 2→0.50, 3→0.80) and a video-level average-risk threshold of 0.8000.

| Metric | Result |
|---|---:|
| comments analyzed | 191,388 |
| Video-level incidents | 1,369 |
| Coverage-review candidates | 41 |
| Candidate rate | 3.0% |
| comments contained in candidate incidents | 57 |
| Fixed benefit per video | KRW 1,000,000 |
| Loading factor | 1.4 |
| Total fixed benefits | KRW 41,000,000 |
| Total gross premium | KRW 57,400,000 |

### Interpretation

The revised mapping raises the baseline risk assigned to score 1 while lowering the maximum risk assigned to score 3. Consequently, the overall mean comment-risk score rises slightly, from 0.4634 to 0.4812, while the extreme-risk tail contracts. At the conservative threshold of 0.8000, candidate incidents decrease from 77 to 41. The model is a screening tool for prioritizing human review, not an automated claim-payment decision system.

### Limitations

1. The manually labeled sample is limited, especially for rare classes such as threat and repetition.
2. Scores for all comments are model predictions and should not be interpreted as equivalent to manual legal review.
3. Results are sensitive to the assumed score-to-probability mapping.
4. Video-level averaging can dilute a small number of high-risk comments in videos with many comments.
5. The 0.8000 threshold identifies review candidates; actual coverage requires policy-based assessment.
6. The model approximates legal risk and does not estimate conviction, litigation success, or settlement probability.
7. Defining one video as one incident may oversimplify cases that span multiple videos or issues.
8. The sample focuses on a limited set of public figures and may not generalize to all online-comment environments.

### Future work

- Expand labeled data and strengthen minority-class representation.
- Compare the five algorithms with fine-tuned Korean language models such as KLUE-BERT.
- Calibrate outcome probabilities using case law, complaints, settlements, and legal-cost data.
- Validate alternative score-to-probability mappings.
- Compare alternative screening rules based on high-risk comment counts, shares, and volume weights.
- Develop issue clustering beyond the one-video-one-incident assumption.


In [ ]:
# Domain-specific processing step; Korean schema keys are retained below.

print('=' * 60)
print('           Full analysis pipeline summary')
print('=' * 60)
print(f'[1]  Source data collection  : {len(df):,} records (After deduplication)' if 'df' in globals() else '[1]  Source data collection  : df missing')
print(f'[2] First-stage LLM filtering   : {len(df_filter):,} records' if 'df_filter' in globals() else '[2] First-stage LLM filtering   : df_filter missing')
if 'df' in globals() and 'df_filter' in globals() and len(df) > 0:
    print(f'    Filtering ratio        : {len(df_filter)/len(df):.1%}')
print(f'[3] Manual labeling     : {len(df_labeled):,} records' if 'df_labeled' in globals() else '[3] Manual labeling     : df_labeled missing')
print(f'[4] Automated model scoring    : {len(scored_df):,} records' if 'scored_df' in globals() else '[4] Automated model scoring    : scored_df missing')
print('[5] Offense probability conversion  : four offenses')
if 'incident_flat_df' in globals():
    print(f'[6] Incident-level aggregation    : {len(incident_flat_df):,} records')
    print(f'[7] Coverage candidate    : {incident_flat_df["is_covered"].sum():,} records ({incident_flat_df["is_covered"].mean():.1%})')
    print(f'[8] total gross premium     : {incident_flat_df["gross_premium"].sum():,.0f} KRW')
else:
    print('[6] Incident-level aggregation    : incident_flat_df missing')
print('=' * 60)


---


In [ ]:
# Domain-specific processing step; Korean schema keys are retained below.

from pathlib import Path
import json

output_xlsx = RISK_DIR / 'final_risk_premium_summary_5algov2.xlsx'

required_vars = ['scored_df', 'incident_flat_df']
missing_vars = [v for v in required_vars if v not in globals()]

if missing_vars:
    print(f'Required DataFrames are missing: {missing_vars}')
    print('Load all prediction data and run the baseline scenario cell first.')
else:
    # -----------------------------
    # Domain-specific processing step; Korean schema keys are retained below.
    # -----------------------------
    total_comments = len(scored_df)
    total_incidents = len(incident_flat_df)
    covered_incidents = int(incident_flat_df['is_covered'].sum())
    covered_ratio = covered_incidents / total_incidents if total_incidents > 0 else 0

    covered_comments = incident_flat_df.loc[
        incident_flat_df['is_covered'],
        'comment_count'
    ].sum()

    summary_df = pd.DataFrame({
        'item': [
            'Analyzed comments',
            'Incident (video) count',
            'Coverage candidate incidents',
            'Coverage-candidate ratio',
            'Comments in coverage-candidate incidents',
            'Risk-score threshold',
            'Fixed benefit limit per video',
            'Loading factor',
            'Maximum gross premium per video',
            'Total fixed benefit',
            'total gross premium',
        ],
        'value': [
            total_comments,
            total_incidents,
            covered_incidents,
            covered_ratio,
            covered_comments,
            RISK_THRESHOLD,
            FIXED_BENEFIT_PER_VIDEO,
            LOADING_FACTOR,
            FIXED_BENEFIT_PER_VIDEO * LOADING_FACTOR,
            incident_flat_df['fixed_benefit'].sum(),
            incident_flat_df['gross_premium'].sum(),
        ],
        'notes': [
            'Based on comments_all_predicted_5algo.csv',
            'video_id one ID = one incident',
            'avg_risk_score >= threshold',
            'Coverage-candidate incidents / all incidents',
            'Comments in coverage-candidate videos',
            'Video-level average-risk threshold',
            'Fixed benefit per coverage-candidate video',
            'Safety loading and operating expenses',
            'Fixed benefit limit × Loading factor',
            'coverage candidate incidents × Fixed benefit limit',
            'Total fixed benefit × Loading factor',
        ],
    })

    # -----------------------------
    # Domain-specific processing step; Korean schema keys are retained below.
    # -----------------------------
    incident_export = incident_flat_df.copy()

    # Domain-specific processing step; Korean schema keys are retained below.
    incident_cols = [
        'incident_rank',
        'video_id',
        'video_title',
        'person',
        'comment_count',
        'avg_risk_score',
        'max_risk_score',
        'avg_total_score',
        'is_covered',
        'fixed_benefit',
        'gross_premium',
    ]

    incident_cols = [c for c in incident_cols if c in incident_export.columns]
    incident_export = incident_export[incident_cols]

    # -----------------------------
    # Domain-specific processing step; Korean schema keys are retained below.
    # -----------------------------
    person_summary = (
        incident_flat_df
        .groupby('person', as_index=False)
        .agg(
            incident_count=('video_id', 'size'),
            covered_incident_count=('is_covered', 'sum'),
            total_comments=('comment_count', 'sum'),
            covered_comments=('comment_count', lambda s: s[incident_flat_df.loc[s.index, 'is_covered']].sum()),
            avg_risk_score=('avg_risk_score', 'mean'),
            max_risk_score=('max_risk_score', 'max'),
            total_fixed_benefit=('fixed_benefit', 'sum'),
            total_gross_premium=('gross_premium', 'sum'),
            avg_gross_premium=('gross_premium', 'mean'),
        )
        .sort_values('total_gross_premium', ascending=False)
    )

    person_summary['covered_incident_ratio'] = (
        person_summary['covered_incident_count'] / person_summary['incident_count']
    )

    # -----------------------------
    # Domain-specific processing step; Korean schema keys are retained below.
    # -----------------------------
    thresholds = [0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]

    threshold_df = pd.DataFrame({
        'threshold': thresholds,
        'covered_incidents': [
            (incident_flat_df['avg_risk_score'] >= t).sum()
            for t in thresholds
        ],
    })

    threshold_df['covered_ratio'] = (
        threshold_df['covered_incidents'] / len(incident_flat_df)
    )
    threshold_df['fixed_benefit_total'] = (
        threshold_df['covered_incidents'] * FIXED_BENEFIT_PER_VIDEO
    )
    threshold_df['gross_premium_total'] = (
        threshold_df['fixed_benefit_total'] * LOADING_FACTOR
    )

    # -----------------------------
    # Domain-specific processing step; Korean schema keys are retained below.
    # -----------------------------
    offense_prob_df = pd.DataFrame({
        'offense': ['insult', 'defamation', 'threat', 'obscene_communication'],
        'average_legal_action_probability': [
            scored_df['p_insult'].mean(),
            scored_df['p_defamation'].mean(),
            scored_df['p_threat'].mean(),
            scored_df['p_obscene_communication'].mean(),
        ],
        'expected_qualifying_count': [
            scored_df['p_insult'].sum(),
            scored_df['p_defamation'].sum(),
            scored_df['p_threat'].sum(),
            scored_df['p_obscene_communication'].sum(),
        ],
    })

    # -----------------------------
    # Domain-specific processing step; Korean schema keys are retained below.
    # -----------------------------
    covered_incidents_df = (
        incident_export[incident_export['is_covered']]
        .sort_values('avg_risk_score', ascending=False)
        .reset_index(drop=True)
    )

    # -----------------------------
    # Domain-specific processing step; Korean schema keys are retained below.
    # -----------------------------
    with pd.ExcelWriter(output_xlsx, engine='openpyxl') as writer:
        summary_df.to_excel(writer, sheet_name='key_summary', index=False)
        incident_export.to_excel(writer, sheet_name='incident_detail', index=False)
        covered_incidents_df.to_excel(writer, sheet_name='coverage_candidate', index=False)
        person_summary.to_excel(writer, sheet_name='subject_summary', index=False)
        threshold_df.to_excel(writer, sheet_name='threshold_sensitivity', index=False)
        offense_prob_df.to_excel(writer, sheet_name='offense_average_probability', index=False)

    print(f'Final Excel file saved: {output_xlsx}')
    print(f'Sheets: key_summary, incident_detail, coverage_candidate, subject_summary, threshold_sensitivity, offense_average_probability')


---
## 11. premium Formula under the Revised Probability Scenario

Changing the probability mapping requires recalculating the entire pipeline, not only the final premium.

1. Convert scores using q(1)=0.20, q(2)=0.50, and q(3)=0.80.
2. Multiply the required legal-element probabilities for each offense.
3. Calculate comment risk as $Risk_j = 1 - \prod_k(1-P_{k,j})$.
4. Average comment risks within each video.
5. Flag videos with $AvgRisk_i \ge 0.8000$.
6. Apply a fixed benefit of KRW 1,000,000 and a loading factor of 1.4.

Under this scenario, 41 candidate incidents produce KRW 41,000,000 in fixed benefits and KRW 57,400,000 in gross premium, or approximately KRW 41,928 per incident across all 1,369 incidents.


In [ ]:
# ============================================================
# Domain-specific processing step; Korean schema keys are retained below.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

# Domain-specific processing step; Korean schema keys are retained below.
# Domain-specific processing step; Korean schema keys are retained below.
# Domain-specific processing step; Korean schema keys are retained below.
# Domain-specific processing step; Korean schema keys are retained below.
# Domain-specific processing step; Korean schema keys are retained below.
# Domain-specific processing step; Korean schema keys are retained below.

BASE = Path.cwd() if 'BASE' not in globals() else BASE
RISK_DIR = BASE / 'Risk_final' if 'RISK_DIR' not in globals() else RISK_DIR
pred_path = RISK_DIR / 'comments_all_predicted_5algov2.csv'

if 'scored_df' in globals():
    scenario_source_df = scored_df.copy()
else:
    scenario_source_df = pd.read_csv(pred_path, encoding='utf-8-sig', low_memory=False)

score_cols_case = [
    'pred_profanity_intensity',
    'pred_factual_assertion',
    'pred_sexual_expression',
    'pred_threat_level',
    'pred_repetition',
    'pred_identifiability',
    'pred_publicity',
]

for col in score_cols_case:
    scenario_source_df[col] = (
        pd.to_numeric(scenario_source_df[col], errors='coerce')
        .fillna(1)
        .clip(1, 3)
        .astype(int)
    )

if 'pred_total_score' not in scenario_source_df.columns:
    scenario_source_df['pred_total_score'] = scenario_source_df[score_cols_case].sum(axis=1)
else:
    scenario_source_df['pred_total_score'] = pd.to_numeric(
        scenario_source_df['pred_total_score'], errors='coerce'
    ).fillna(scenario_source_df[score_cols_case].sum(axis=1))

if 'person' not in scenario_source_df.columns:
    SUBJECT_KEYWORDS = {
    'Subject_A': ['REDACTED_SUBJECT_A'],
    'Subject_B': ['REDACTED_SUBJECT_B'],
    'Subject_C': ['REDACTED_SUBJECT_C'],
}

    def tag_person_case(title):
        t = str(title).lower()
        for person, keywords in SUBJECT_KEYWORDS.items():
            if any(keyword.lower() in t for keyword in keywords):
                return person
        return 'Other'

    scenario_source_df['person'] = scenario_source_df['video_title'].map(tag_person_case)

SCENARIO_PROB_MAPS = {
    'baseline_0.10_0.50_0.90': {1: 0.10, 2: 0.50, 3: 0.90},
    'revised_0.20_0.50_0.80': {1: 0.20, 2: 0.50, 3: 0.80},
}

RISK_THRESHOLD_CASE = globals().get('RISK_THRESHOLD', 0.8000)
FIXED_BENEFIT_CASE = globals().get('FIXED_BENEFIT_PER_VIDEO', 1_000_000)
LOADING_FACTOR_CASE = globals().get('LOADING_FACTOR', 1.4)
THRESHOLD_GRID = [0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]

scenario_summary_rows = []
scenario_offense_rows = []
scenario_threshold_rows = []
scenario_person_frames = []
scenario_incident_frames = []
scenario_covered_frames = []

for scenario_name, prob_map in SCENARIO_PROB_MAPS.items():
    df_case = scenario_source_df.copy()

    def p_score(col):
        return df_case[col].map(prob_map)

    # Domain-specific processing step; Korean schema keys are retained below.
    df_case['p_insult'] = (
        p_score('pred_profanity_intensity') *
        p_score('pred_identifiability') *
        p_score('pred_publicity')
    )
    df_case['p_defamation'] = (
        p_score('pred_factual_assertion') *
        p_score('pred_identifiability') *
        p_score('pred_publicity')
    )
    df_case['p_threat'] = (
        p_score('pred_threat_level') *
        p_score('pred_identifiability')
    )
    df_case['p_obscene_communication'] = (
        p_score('pred_sexual_expression') *
        p_score('pred_identifiability') *
        p_score('pred_publicity')
    )

    offense_cols = ['p_insult', 'p_defamation', 'p_threat', 'p_obscene_communication']
    offense_names = ['insult', 'defamation', 'threat', 'obscene_communication']

    # Domain-specific processing step; Korean schema keys are retained below.
    df_case['comment_risk_score'] = 1 - np.prod(
        [(1 - df_case[col]) for col in offense_cols],
        axis=0,
    )

    # Domain-specific processing step; Korean schema keys are retained below.
    incident_case = (
        df_case
        .groupby(['video_id', 'video_title', 'person'], as_index=False)
        .agg(
            comment_count=('text', 'size'),
            avg_risk_score=('comment_risk_score', 'mean'),
            max_risk_score=('comment_risk_score', 'max'),
            avg_total_score=('pred_total_score', 'mean'),
            mean_p_insult=('p_insult', 'mean'),
            mean_p_defamation=('p_defamation', 'mean'),
            mean_p_threat=('p_threat', 'mean'),
            mean_p_obscene_communication=('p_obscene_communication', 'mean'),
            max_p_insult=('p_insult', 'max'),
            max_p_defamation=('p_defamation', 'max'),
            max_p_threat=('p_threat', 'max'),
            max_p_obscene_communication=('p_obscene_communication', 'max'),
        )
    )
    incident_case['scenario'] = scenario_name
    incident_case['risk_threshold'] = RISK_THRESHOLD_CASE
    incident_case['is_covered'] = incident_case['avg_risk_score'] >= RISK_THRESHOLD_CASE
    incident_case['fixed_benefit'] = np.where(
        incident_case['is_covered'],
        FIXED_BENEFIT_CASE,
        0,
    )
    incident_case['gross_premium'] = incident_case['fixed_benefit'] * LOADING_FACTOR_CASE
    incident_case = incident_case.sort_values(
        ['is_covered', 'avg_risk_score'],
        ascending=[False, False],
    ).reset_index(drop=True)
    incident_case.insert(0, 'incident_rank', np.arange(1, len(incident_case) + 1))

    total_comments = len(df_case)
    total_incidents = len(incident_case)
    covered_mask = incident_case['is_covered']
    covered_incidents = int(covered_mask.sum())
    covered_comments = int(incident_case.loc[covered_mask, 'comment_count'].sum())

    scenario_summary_rows.append({
        'scenario': scenario_name,
        'score_1_probability': prob_map[1],
        'score_2_probability': prob_map[2],
        'score_3_probability': prob_map[3],
        'risk_threshold': RISK_THRESHOLD_CASE,
        'analyzed_comment_count': total_comments,
        'incident_count': total_incidents,
        'coverage_candidate_incident_count': covered_incidents,
        'coverage_candidate_ratio': covered_incidents / total_incidents if total_incidents else 0,
        'covered_candidate_comment_count': covered_comments,
        'fixed_benefit_limit_per_video': FIXED_BENEFIT_CASE,
        'loading_factor': LOADING_FACTOR_CASE,
        'maximum_gross_premium_per_video': int(round(FIXED_BENEFIT_CASE * LOADING_FACTOR_CASE)),
        'total_fixed_benefit': int(incident_case['fixed_benefit'].sum()),
        'total_gross_premium': int(round(incident_case['gross_premium'].sum())),
        'average_gross_premium_per_incident': incident_case['gross_premium'].mean(),
        'median_gross_premium_per_incident': incident_case['gross_premium'].median(),
        'average_comment_risk_score': df_case['comment_risk_score'].mean(),
        'average_incident_risk_score': incident_case['avg_risk_score'].mean(),
    })

    for offense, col in zip(offense_names, offense_cols):
        scenario_offense_rows.append({
            'scenario': scenario_name,
            'offense': offense,
            'average_legal_action_probability': df_case[col].mean(),
            'expected_qualifying_comment_count': df_case[col].sum(),
            'p>=0.2_comment_count': int((df_case[col] >= 0.2).sum()),
            'p>=0.5_comment_count': int((df_case[col] >= 0.5).sum()),
            'p>=0.8_comment_count': int((df_case[col] >= 0.8).sum()),
            'mean_incident_average_probability': incident_case[f'mean_p_{offense}'].mean(),
            'mean_incident_max_probability': incident_case[f'max_p_{offense}'].mean(),
        })

    for threshold in THRESHOLD_GRID:
        threshold_mask = incident_case['avg_risk_score'] >= threshold
        scenario_threshold_rows.append({
            'scenario': scenario_name,
            'threshold': threshold,
            'covered_incidents': int(threshold_mask.sum()),
            'covered_ratio': threshold_mask.mean(),
            'covered_comments': int(incident_case.loc[threshold_mask, 'comment_count'].sum()),
            'fixed_benefit_total': int(threshold_mask.sum() * FIXED_BENEFIT_CASE),
            'gross_premium_total': int(round(threshold_mask.sum() * FIXED_BENEFIT_CASE * LOADING_FACTOR_CASE)),
        })

    person_case = (
        incident_case
        .groupby('person', as_index=False)
        .agg(
            incident_count=('video_id', 'size'),
            covered_incident_count=('is_covered', 'sum'),
            total_comments=('comment_count', 'sum'),
            avg_risk_score=('avg_risk_score', 'mean'),
            max_risk_score=('max_risk_score', 'max'),
            total_fixed_benefit=('fixed_benefit', 'sum'),
            total_gross_premium=('gross_premium', 'sum'),
            avg_gross_premium=('gross_premium', 'mean'),
        )
    )
    covered_comments_by_person = (
        incident_case.loc[incident_case['is_covered']]
        .groupby('person')['comment_count']
        .sum()
    )
    person_case['covered_comments'] = person_case['person'].map(covered_comments_by_person).fillna(0).astype(int)
    person_case['covered_incident_ratio'] = person_case['covered_incident_count'] / person_case['incident_count']
    person_case['scenario'] = scenario_name
    scenario_person_frames.append(person_case.sort_values('total_gross_premium', ascending=False))

    scenario_incident_frames.append(incident_case)
    scenario_covered_frames.append(incident_case.loc[incident_case['is_covered']].copy())

scenario_summary = pd.DataFrame(scenario_summary_rows)
scenario_offense_summary = pd.DataFrame(scenario_offense_rows)
scenario_threshold_summary = pd.DataFrame(scenario_threshold_rows)
scenario_person_summary = pd.concat(scenario_person_frames, ignore_index=True)
scenario_incident_detail = pd.concat(scenario_incident_frames, ignore_index=True)

# Domain-specific processing step; Korean schema keys are retained below.
premium_formula_df = pd.DataFrame([
    {
        'stage': '1. Score-to-probability mapping',
        'formula': 'q(1)=0.20, q(2)=0.50, q(3)=0.80',
        'description': 'Convert predicted scores of 1/2/3 to probabilities in the revised scenario',
    },
    {
        'stage': '2. by offense legal-action probability',
        'formula': 'P(offense) = product of q(legal-element score)',
        'description': 'Multiply legal-element probabilities for insult, defamation, threat, and obscene communication',
    },
    {
        'stage': '3. Comment risk score',
        'formula': 'Risk_j = 1 - Π(1 - P_{k,j})',
        'description': 'Probability that at least one of the four offenses applies to comment j',
    },
    {
        'stage': '4. Average video risk score',
        'formula': 'AvgRisk_i = mean(Risk_j), j in video i',
        'description': 'Average comment risk scores by video_id',
    },
    {
        'stage': '5. Coverage-candidate determination',
        'formula': 'Covered_i = 1 if AvgRisk_i >= 0.8000 else 0',
        'description': 'Only videos with a score of at least 0.8000 are coverage candidates',
    },
    {
        'stage': '6. incident-level Fixed benefit',
        'formula': 'Benefit_i = Covered_i × 1,000,000 KRW',
        'description': 'Do not accumulate comment-level benefits within the same video',
    },
    {
        'stage': '7. incident-level gross premium',
        'formula': 'Grosspremium_i = Benefit_i × 1.4',
        'description': 'Apply a 1.4 loading factor for safety margin and operating expenses',
    },
    {
        'stage': '8. total gross premium',
        'formula': 'TotalGrosspremium = Σ Grosspremium_i',
        'description': 'Under the revised scenario 41 records × 1,000,000 KRW × 1.4 = 57,400,000 KRW',
    },
])

scenario_covered_detail = pd.concat(scenario_covered_frames, ignore_index=True)

# Domain-specific processing step; Korean schema keys are retained below.
scenario_output_xlsx = RISK_DIR / 'probability_scenario_full_premium_020_050_080.xlsx'
final_output_xlsx = RISK_DIR / 'final_risk_premium_summary_5algov2.xlsx'

with pd.ExcelWriter(scenario_output_xlsx, engine='openpyxl') as writer:
    premium_formula_df.to_excel(writer, sheet_name='premium_formula', index=False)
    scenario_summary.to_excel(writer, sheet_name='scenario_key_summary', index=False)
    scenario_offense_summary.to_excel(writer, sheet_name='scenario_offense_probability', index=False)
    scenario_threshold_summary.to_excel(writer, sheet_name='scenario_threshold', index=False)
    scenario_person_summary.to_excel(writer, sheet_name='scenario_subject', index=False)
    scenario_covered_detail.to_excel(writer, sheet_name='scenario_coverage_candidates', index=False)
    scenario_incident_detail.to_excel(writer, sheet_name='scenario_incident_detail', index=False)

if final_output_xlsx.exists():
    with pd.ExcelWriter(final_output_xlsx, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        premium_formula_df.to_excel(writer, sheet_name='premium_formula', index=False)
        scenario_summary.to_excel(writer, sheet_name='probability_scenario_key_summary', index=False)
        scenario_offense_summary.to_excel(writer, sheet_name='probability_scenario_offense_probability', index=False)
        scenario_threshold_summary.to_excel(writer, sheet_name='probability_scenario_threshold', index=False)
        scenario_person_summary.to_excel(writer, sheet_name='probability_scenario_subject', index=False)

print('Full premium recalculation by probability scenario complete')
print(f'- Separately saved: {scenario_output_xlsx}')
if final_output_xlsx.exists():
    print(f'- Add or replace sheets in the final summary file: {final_output_xlsx}')

print('\n[Key summary by probability scenario]')
display(scenario_summary)

print('\n[By probability scenario by offense legal-action probability]')
display(scenario_offense_summary)

print('\n[Threshold sensitivity by probability scenario]')
display(scenario_threshold_summary)


---
## 12. Probability-Scenario Visualization

The charts compare candidate incident counts, total gross premium, threshold sensitivity, offense-level probabilities, and person-level premiums under alternative probability mappings.


In [ ]:
# ============================================================
# Domain-specific processing step; Korean schema keys are retained below.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager
try:
    from IPython.display import display as notebook_display, Image as NotebookImage
except Exception:
    notebook_display = None
    NotebookImage = None

try:
    display
except NameError:
    display = print

BASE = Path.cwd() if 'BASE' not in globals() else BASE
RISK_DIR = BASE / 'Risk_final' if 'RISK_DIR' not in globals() else RISK_DIR
CHART_DIR = RISK_DIR / 'scenario_charts'
CHART_DIR.mkdir(parents=True, exist_ok=True)

scenario_output_xlsx = RISK_DIR / 'probability_scenario_full_premium_020_050_080.xlsx'
if not scenario_output_xlsx.exists():
    raise FileNotFoundError('Run the full premium recalculation cell for each probability scenario first.')

scenario_summary = pd.read_excel(scenario_output_xlsx, sheet_name='scenario_key_summary')
scenario_offense_summary = pd.read_excel(scenario_output_xlsx, sheet_name='scenario_offense_probability')
scenario_threshold_summary = pd.read_excel(scenario_output_xlsx, sheet_name='scenario_threshold')
scenario_person_summary = pd.read_excel(scenario_output_xlsx, sheet_name='scenario_subject')
scenario_covered_detail = pd.read_excel(scenario_output_xlsx, sheet_name='scenario_coverage_candidates')

# Configure a Korean-capable font for source labels.
font_candidates = [
    '/System/Library/Fonts/AppleSDGothicNeo.ttc',
    '/System/Library/Fonts/Supplemental/AppleGothic.ttf',
    '/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
    '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc',
]
selected_font = None
for font_path in font_candidates:
    if Path(font_path).exists():
        try:
            font_manager.fontManager.addfont(font_path)
            selected_font = font_manager.FontProperties(fname=font_path).get_name()
            break
        except Exception:
            continue

if selected_font:
    plt.rcParams['font.family'] = selected_font
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

scenario_order = ['baseline_0.10_0.50_0.90', 'revised_0.20_0.50_0.80']
scenario_label = {
    'baseline_0.10_0.50_0.90': 'baseline\n0.10/0.50/0.90',
    'revised_0.20_0.50_0.80': 'revised\n0.20/0.50/0.80',
}
colors = {
    'baseline_0.10_0.50_0.90': '#4C72B0',
    'revised_0.20_0.50_0.80': '#C44E52',
}

summary_plot = scenario_summary.set_index('scenario').loc[scenario_order].reset_index()

# ------------------------------------------------------------
# Domain-specific processing step; Korean schema keys are retained below.
# ------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Premium calculation summary by probability scenario', fontsize=16, fontweight='bold')

x = np.arange(len(summary_plot))
bar_colors = [colors[s] for s in summary_plot['scenario']]
labels = [scenario_label[s] for s in summary_plot['scenario']]

ax = axes[0, 0]
bars = ax.bar(x, summary_plot['coverage_candidate_incident_count'], color=bar_colors)
ax.set_title('Coverage candidate incidents')
ax.set_xticks(x, labels)
ax.set_ylabel('incidents')
for b, v in zip(bars, summary_plot['coverage_candidate_incident_count']):
    ax.text(b.get_x() + b.get_width()/2, b.get_height(), f'{int(v):,} records', ha='center', va='bottom', fontsize=10)
ax.grid(axis='y', alpha=0.25)

ax = axes[0, 1]
gross_m = summary_plot['total_gross_premium'] / 1e6
bars = ax.bar(x, gross_m, color=bar_colors)
ax.set_title('total gross premium')
ax.set_xticks(x, labels)
ax.set_ylabel('KRW millions')
for b, v in zip(bars, gross_m):
    ax.text(b.get_x() + b.get_width()/2, b.get_height(), f'{v:.1f}M', ha='center', va='bottom', fontsize=10)
ax.grid(axis='y', alpha=0.25)

ax = axes[1, 0]
bars = ax.bar(x, summary_plot['average_incident_risk_score'], color=bar_colors)
ax.axhline(0.8, color='#333333', linestyle='--', linewidth=1.2, label='threshold 0.8')
ax.set_title('Average incident risk score')
ax.set_xticks(x, labels)
ax.set_ylim(0, max(0.9, summary_plot['average_incident_risk_score'].max() * 1.25))
for b, v in zip(bars, summary_plot['average_incident_risk_score']):
    ax.text(b.get_x() + b.get_width()/2, b.get_height(), f'{v:.3f}', ha='center', va='bottom', fontsize=10)
ax.legend(frameon=False)
ax.grid(axis='y', alpha=0.25)

ax = axes[1, 1]
bars = ax.bar(x, summary_plot['covered_candidate_comment_count'], color=bar_colors)
ax.set_title('Comments in coverage-candidate incidents')
ax.set_xticks(x, labels)
ax.set_ylabel('comments')
for b, v in zip(bars, summary_plot['covered_candidate_comment_count']):
    ax.text(b.get_x() + b.get_width()/2, b.get_height(), f'{int(v):,} records', ha='center', va='bottom', fontsize=10)
ax.grid(axis='y', alpha=0.25)

plt.tight_layout(rect=[0, 0, 1, 0.95])
dashboard_path = CHART_DIR / 'scenario_premium_dashboard.png'
plt.savefig(dashboard_path, bbox_inches='tight')
plt.close(fig)

# ------------------------------------------------------------
# Domain-specific processing step; Korean schema keys are retained below.
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Threshold sensitivity analysis', fontsize=15, fontweight='bold')

for scenario in scenario_order:
    sub = scenario_threshold_summary[scenario_threshold_summary['scenario'] == scenario].sort_values('threshold')
    axes[0].plot(sub['threshold'], sub['covered_incidents'], marker='o', linewidth=2, color=colors[scenario], label=scenario_label[scenario].replace('\n', ' '))
    axes[1].plot(sub['threshold'], sub['gross_premium_total'] / 1e6, marker='o', linewidth=2, color=colors[scenario], label=scenario_label[scenario].replace('\n', ' '))

for ax in axes:
    ax.axvline(0.8, color='#333333', linestyle='--', linewidth=1.1)
    ax.grid(alpha=0.25)
    ax.set_xlabel('Video-level average-risk threshold')
    ax.legend(frameon=False)

axes[0].set_title('Coverage candidate incidents')
axes[0].set_ylabel('incidents')
axes[1].set_title('total gross premium')
axes[1].set_ylabel('KRW millions')

plt.tight_layout(rect=[0, 0, 1, 0.93])
threshold_path = CHART_DIR / 'scenario_threshold_sensitivity.png'
plt.savefig(threshold_path, bbox_inches='tight')
plt.close(fig)

# ------------------------------------------------------------
# Domain-specific processing step; Korean schema keys are retained below.
# ------------------------------------------------------------
offense_order = ['insult', 'defamation', 'threat', 'obscene_communication']
pivot_offense = (
    scenario_offense_summary
    .pivot(index='offense', columns='scenario', values='average_legal_action_probability')
    .loc[offense_order, scenario_order]
)

fig, ax = plt.subplots(figsize=(10, 5.5))
x = np.arange(len(pivot_offense))
width = 0.36
for idx, scenario in enumerate(scenario_order):
    vals = pivot_offense[scenario].values
    offset = (idx - 0.5) * width
    bars = ax.bar(x + offset, vals, width=width, color=colors[scenario], label=scenario_label[scenario].replace('\n', ' '))
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, b.get_height(), f'{v:.3f}', ha='center', va='bottom', fontsize=9)

ax.set_title('Average complaint-risk probability by offense')
ax.set_xticks(x, pivot_offense.index)
ax.set_ylabel('Average probability')
ax.set_ylim(0, max(0.3, pivot_offense.values.max() * 1.25))
ax.legend(frameon=False)
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
offense_path = CHART_DIR / 'scenario_offense_probability.png'
plt.savefig(offense_path, bbox_inches='tight')
plt.close(fig)

# ------------------------------------------------------------
# Domain-specific processing step; Korean schema keys are retained below.
# ------------------------------------------------------------
person_order = (
    scenario_person_summary
    .groupby('person')['total_gross_premium']
    .max()
    .sort_values(ascending=False)
    .index
    .tolist()
)
pivot_person = (
    scenario_person_summary
    .pivot(index='person', columns='scenario', values='total_gross_premium')
    .reindex(person_order)[scenario_order]
    .fillna(0)
)

fig, ax = plt.subplots(figsize=(10, 5.5))
x = np.arange(len(pivot_person))
width = 0.36
for idx, scenario in enumerate(scenario_order):
    vals = pivot_person[scenario].values / 1e6
    offset = (idx - 0.5) * width
    bars = ax.bar(x + offset, vals, width=width, color=colors[scenario], label=scenario_label[scenario].replace('\n', ' '))
    for b, v in zip(bars, vals):
        if v > 0:
            ax.text(b.get_x() + b.get_width()/2, b.get_height(), f'{v:.1f}M', ha='center', va='bottom', fontsize=9)

ax.set_title('by subject total gross premium')
ax.set_xticks(x, pivot_person.index)
ax.set_ylabel('KRW millions')
ax.legend(frameon=False)
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
person_path = CHART_DIR / 'scenario_person_total_premium.png'
plt.savefig(person_path, bbox_inches='tight')
plt.close(fig)

# ------------------------------------------------------------
# Domain-specific processing step; Korean schema keys are retained below.
# ------------------------------------------------------------
changed_covered = scenario_covered_detail[scenario_covered_detail['scenario'] == 'revised_0.20_0.50_0.80'].copy()

fig, ax = plt.subplots(figsize=(9, 5))
if len(changed_covered) > 0:
    ax.hist(changed_covered['avg_risk_score'], bins=12, color='#C44E52', alpha=0.85, edgecolor='white')
    ax.axvline(0.8, color='#333333', linestyle='--', linewidth=1.2, label='threshold 0.8')
    ax.set_title('Average-risk distribution of coverage-candidate incidents in the revised scenario')
    ax.set_xlabel('Video-level average risk score')
    ax.set_ylabel('incidents')
    ax.legend(frameon=False)
    ax.grid(axis='y', alpha=0.25)
else:
    ax.text(0.5, 0.5, 'No coverage-review candidate incidents', ha='center', va='center')
    ax.set_axis_off()
plt.tight_layout()
covered_dist_path = CHART_DIR / 'changed_scenario_covered_risk_distribution.png'
plt.savefig(covered_dist_path, bbox_inches='tight')
plt.close(fig)

chart_paths = pd.DataFrame({
    'chart': [
        'Scenario key dashboard',
        'Threshold sensitivity',
        'Average complaint-risk probability by offense',
        'by subject total gross premium',
        'Coverage-candidate risk distribution in the revised scenario',
    ],
    'file': [
        str(dashboard_path),
        str(threshold_path),
        str(offense_path),
        str(person_path),
        str(covered_dist_path),
    ],
})

# Domain-specific processing step; Korean schema keys are retained below.
scenario_output_xlsx = RISK_DIR / 'probability_scenario_full_premium_020_050_080.xlsx'
final_output_xlsx = RISK_DIR / 'final_risk_premium_summary_5algov2.xlsx'
for workbook_path in [scenario_output_xlsx, final_output_xlsx]:
    if workbook_path.exists():
        with pd.ExcelWriter(workbook_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
            chart_paths.to_excel(writer, sheet_name='visualization_index', index=False)

# Domain-specific processing step; Korean schema keys are retained below.
if notebook_display is not None and NotebookImage is not None:
    for _, row in chart_paths.iterrows():
        notebook_display(row['chart'])
        notebook_display(NotebookImage(filename=row['file']))

display(chart_paths)
print('Visualizations saved')
print(f'Output directory: {CHART_DIR}')


---
## 13. Final premium Summary

This section summarizes per-person and per-video premiums and average offense-level complaint probabilities under the revised 0.20 / 0.50 / 0.80 mapping.


In [ ]:
# ============================================================
# Domain-specific processing step; Korean schema keys are retained below.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager

try:
    from IPython.display import display as notebook_display, Image as NotebookImage
except Exception:
    notebook_display = None
    NotebookImage = None

try:
    display
except NameError:
    display = print

BASE = Path.cwd() if 'BASE' not in globals() else BASE
RISK_DIR = BASE / 'Risk_final' if 'RISK_DIR' not in globals() else RISK_DIR
CHART_DIR = RISK_DIR / 'scenario_charts'
CHART_DIR.mkdir(parents=True, exist_ok=True)

scenario_output_xlsx = RISK_DIR / 'probability_scenario_full_premium_020_050_080.xlsx'
final_output_xlsx = RISK_DIR / 'final_risk_premium_summary_5algov2.xlsx'
if not scenario_output_xlsx.exists():
    raise FileNotFoundError('Run the full premium recalculation cell for each probability scenario first.')

SCENARIO_NAME = 'revised_0.20_0.50_0.80'
FIXED_BENEFIT_CASE = 1_000_000
LOADING_FACTOR_CASE = 1.4

scenario_summary_all = pd.read_excel(scenario_output_xlsx, sheet_name='scenario_key_summary')
incident_all = pd.read_excel(scenario_output_xlsx, sheet_name='scenario_incident_detail')
offense_all = pd.read_excel(scenario_output_xlsx, sheet_name='scenario_offense_probability')

scenario_summary = scenario_summary_all[scenario_summary_all['scenario'] == SCENARIO_NAME].copy()
incident_df = incident_all[incident_all['scenario'] == SCENARIO_NAME].copy()
offense_probability_summary = offense_all[offense_all['scenario'] == SCENARIO_NAME].copy()

if incident_df.empty:
    raise ValueError(f'{SCENARIO_NAME} has no incident-level scenario data.')

# Configure a Korean-capable font for source labels.
font_candidates = [
    '/System/Library/Fonts/AppleSDGothicNeo.ttc',
    '/System/Library/Fonts/Supplemental/AppleGothic.ttf',
    '/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
    '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc',
]
selected_font = None
for font_path in font_candidates:
    if Path(font_path).exists():
        try:
            font_manager.fontManager.addfont(font_path)
            selected_font = font_manager.FontProperties(fname=font_path).get_name()
            break
        except Exception:
            continue
if selected_font:
    plt.rcParams['font.family'] = selected_font
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

# ------------------------------------------------------------
# Domain-specific processing step; Korean schema keys are retained below.
# ------------------------------------------------------------
covered_comments_by_person = (
    incident_df.loc[incident_df['is_covered']]
    .groupby('person')['comment_count']
    .sum()
)

person_video_premium_final = (
    incident_df
    .groupby('person', as_index=False)
    .agg(
        video_count=('video_id', 'size'),
        coverage_candidate_video_count=('is_covered', 'sum'),
        total_comment_count=('comment_count', 'sum'),
        average_risk_score=('avg_risk_score', 'mean'),
        maximum_risk_score=('max_risk_score', 'max'),
        risk_premium_per_video=('fixed_benefit', 'mean'),
        gross_premium_per_video=('gross_premium', 'mean'),
        total_risk_premium=('fixed_benefit', 'sum'),
        total_gross_premium=('gross_premium', 'sum'),
    )
)
person_video_premium_final['covered_candidate_comment_count'] = (
    person_video_premium_final['person']
    .map(covered_comments_by_person)
    .fillna(0)
    .astype(int)
)
person_video_premium_final['coverage_candidate_ratio'] = (
    person_video_premium_final['coverage_candidate_video_count'] /
    person_video_premium_final['video_count']
)
person_video_premium_final['maximum_benefit_per_video'] = FIXED_BENEFIT_CASE
person_video_premium_final = person_video_premium_final.sort_values(
    'total_gross_premium', ascending=False
).reset_index(drop=True)

# ------------------------------------------------------------
# Domain-specific processing step; Korean schema keys are retained below.
# ------------------------------------------------------------
video_one_premium_summary = pd.DataFrame({
    'item': [
        'All videos',
        'Coverage-candidate videos',
        'Coverage-candidate ratio',
        'Comments in coverage candidates',
        'Average risk premium per video',
        'Average gross premium per video',
        'Maximum benefit per video',
        'Maximum gross premium per video',
        'total risk premium',
        'total gross premium',
        'Maximum possible benefit across all videos',
    ],
    'value': [
        len(incident_df),
        int(incident_df['is_covered'].sum()),
        incident_df['is_covered'].mean(),
        int(incident_df.loc[incident_df['is_covered'], 'comment_count'].sum()),
        incident_df['fixed_benefit'].mean(),
        incident_df['gross_premium'].mean(),
        FIXED_BENEFIT_CASE,
        FIXED_BENEFIT_CASE * LOADING_FACTOR_CASE,
        int(round(incident_df['fixed_benefit'].sum())),
        int(round(incident_df['gross_premium'].sum())),
        FIXED_BENEFIT_CASE * len(incident_df),
    ],
    'notes': [
        'Incidents grouped by video_id, video_title, and subject',
        'avg_risk_score >= 0.8000',
        'Coverage-candidate videos / All videos',
        'Comments in coverage-candidate videos',
        'total risk premium / All videos',
        'total gross premium / All videos',
        'Fixed benefit limit per coverage-candidate video',
        'Fixed benefit limit × loading factor 1.4',
        'Coverage-candidate videos × fixed benefit limit',
        'total risk premium × 1.4',
        'Theoretical maximum benefit if every video is covered',
    ],
})

# ------------------------------------------------------------
# Domain-specific processing step; Korean schema keys are retained below.
# ------------------------------------------------------------
offense_probability_final = offense_probability_summary[[
    'offense',
    'average_legal_action_probability',
    'expected_qualifying_comment_count',
    'p>=0.2_comment_count',
    'p>=0.5_comment_count',
    'p>=0.8_comment_count',
    'mean_incident_average_probability',
    'mean_incident_max_probability',
]].copy()
offense_probability_final = offense_probability_final.sort_values(
    'average_legal_action_probability', ascending=False
).reset_index(drop=True)

print('=== Premium per video Summary by subject ===')
display(person_video_premium_final)
print('\n=== Per-video premium summary ===')
display(video_one_premium_summary)
print('\n=== Average complaint-risk probability by offense ===')
display(offense_probability_final)

# ------------------------------------------------------------
# Domain-specific processing step; Korean schema keys are retained below.
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
plot_person = person_video_premium_final.copy()
colors_person = ['#4C72B0', '#DD8452', '#55A467', '#C44E52'][:len(plot_person)]

bars = axes[0].bar(plot_person['person'], plot_person['gross_premium_per_video'] / 1e6, color=colors_person)
axes[0].set_title('by subject Gross premium per video')
axes[0].set_ylabel('KRW millions')
axes[0].grid(axis='y', alpha=0.25)
for b, v in zip(bars, plot_person['gross_premium_per_video'] / 1e6):
    axes[0].text(b.get_x() + b.get_width()/2, b.get_height(), f'{v:.3f}M', ha='center', va='bottom', fontsize=9)

bars = axes[1].bar(plot_person['person'], plot_person['total_gross_premium'] / 1e6, color=colors_person)
axes[1].set_title('by subject total gross premium')
axes[1].set_ylabel('KRW millions')
axes[1].grid(axis='y', alpha=0.25)
for b, v in zip(bars, plot_person['total_gross_premium'] / 1e6):
    axes[1].text(b.get_x() + b.get_width()/2, b.get_height(), f'{v:.1f}M', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
person_final_chart = CHART_DIR / 'final_person_video_premium.png'
plt.savefig(person_final_chart, bbox_inches='tight')
plt.close(fig)

# ------------------------------------------------------------
# Domain-specific processing step; Korean schema keys are retained below.
# ------------------------------------------------------------
video_bar_df = pd.DataFrame({
    'item': ['Average risk premium', 'Average gross premium', 'Maximum benefit', 'Maximum gross premium'],
    'amount': [
        incident_df['fixed_benefit'].mean(),
        incident_df['gross_premium'].mean(),
        FIXED_BENEFIT_CASE,
        FIXED_BENEFIT_CASE * LOADING_FACTOR_CASE,
    ],
})

fig, ax = plt.subplots(figsize=(9, 5.5))
bars = ax.bar(video_bar_df['item'], video_bar_df['amount'] / 1e6, color=['#4C72B0', '#DD8452', '#55A467', '#C44E52'])
ax.set_title('Per-video premium summary')
ax.set_ylabel('KRW millions')
ax.grid(axis='y', alpha=0.25)
ax.tick_params(axis='x', rotation=10)
for b, v in zip(bars, video_bar_df['amount']):
    ax.text(b.get_x() + b.get_width()/2, b.get_height(), f'{v:,.0f} KRW', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
video_final_chart = CHART_DIR / 'final_video_one_premium_summary.png'
plt.savefig(video_final_chart, bbox_inches='tight')
plt.close(fig)

# ------------------------------------------------------------
# Domain-specific processing step; Korean schema keys are retained below.
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 5.5))
offense_plot = offense_probability_final.sort_values('average_legal_action_probability', ascending=True)
bars = ax.barh(offense_plot['offense'], offense_plot['average_legal_action_probability'], color='#4C72B0')
ax.set_title('Average complaint-risk probability by offense')
ax.set_xlabel('Average probability')
ax.grid(axis='x', alpha=0.25)
for b, v in zip(bars, offense_plot['average_legal_action_probability']):
    ax.text(v, b.get_y() + b.get_height()/2, f'{v:.3f}', va='center', ha='left', fontsize=10)
plt.tight_layout()
offense_final_chart = CHART_DIR / 'final_offense_avg_probability.png'
plt.savefig(offense_final_chart, bbox_inches='tight')
plt.close(fig)

# ------------------------------------------------------------
# Domain-specific processing step; Korean schema keys are retained below.
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 5.5))
offense_count_plot = offense_probability_final.sort_values('expected_qualifying_comment_count', ascending=True)
bars = ax.barh(offense_count_plot['offense'], offense_count_plot['expected_qualifying_comment_count'], color='#C44E52')
ax.set_title('expected_qualifying_comment_count_by_offense')
ax.set_xlabel('Expected qualifying comment count')
ax.grid(axis='x', alpha=0.25)
for b, v in zip(bars, offense_count_plot['expected_qualifying_comment_count']):
    ax.text(v, b.get_y() + b.get_height()/2, f'{v:,.0f}', va='center', ha='left', fontsize=10)
plt.tight_layout()
offense_count_chart = CHART_DIR / 'final_offense_expected_count.png'
plt.savefig(offense_count_chart, bbox_inches='tight')
plt.close(fig)

final_chart_paths = pd.DataFrame({
    'chart': [
        'Per-video and total gross premium by subject',
        'Per-video premium summary',
        'Average complaint-risk probability by offense',
        'expected_qualifying_comment_count_by_offense',
    ],
    'file': [
        str(person_final_chart),
        str(video_final_chart),
        str(offense_final_chart),
        str(offense_count_chart),
    ],
})

# Export results to Excel.
for workbook_path in [scenario_output_xlsx, final_output_xlsx]:
    if workbook_path.exists():
        with pd.ExcelWriter(workbook_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
            person_video_premium_final.to_excel(writer, sheet_name='final_subject_video_premium', index=False)
            video_one_premium_summary.to_excel(writer, sheet_name='final_per_video_premium', index=False)
            offense_probability_final.to_excel(writer, sheet_name='final_offense_probability', index=False)
            final_chart_paths.to_excel(writer, sheet_name='final_visualization_index', index=False)

# Display saved charts in the notebook.
if notebook_display is not None and NotebookImage is not None:
    for _, row in final_chart_paths.iterrows():
        notebook_display(row['chart'])
        notebook_display(NotebookImage(filename=row['file']))

display(final_chart_paths)
print('Final summaries and visualizations by subject, video, and offense saved')
print(f'Output directory: {CHART_DIR}')
